# WP3B-M4 — Bounded adaptation with a shared frozen-feature cache

M3 selected epoch1 reaches H@1 14.65% but fails its disease-safety gate: 71 diseased hits versus frozen79, and 873 false-healthy errors versus control490. Do not train it longer. Test one explicit relative residual cap, with all other training choices fixed. Cache only frozen native-CLIP embeddings of the original training-derived folds to avoid repeated encodings. Neither results nor speedups are claimed until owner execution. All essential processing is visible here.


In [ ]:
from pathlib import Path
from dataclasses import dataclass
from collections import Counter
from numbers import Integral, Real
from typing import Any
from datetime import datetime, timezone
import ast
import hashlib
import importlib.metadata as metadata
import json
import math
import platform
import random
import sys
import time
import unicodedata
import uuid
import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for directory in (start.resolve(), *start.resolve().parents):
        if (directory / "DPD_Small_train.csv").is_file() and (directory / "notebooks").is_dir():
            return directory
    raise FileNotFoundError("Open this notebook inside the WSL repository containing DPD_Small_train.csv.")
ROOT = find_project_root()
print({"python": platform.python_version(), "torch": str(torch.__version__),
       "torchvision": str(torchvision.__version__), "cuda_available": torch.cuda.is_available(),
       "repository": str(ROOT)})


## 2. Default-off execution controls and proposed pilot
One proposed configuration: seed 42, native 224 CLIP images without augmentation, image batch 8, text batch 32, AdamW LR 1e-4 / decay 1e-4, gradient-norm clip 1, fixed five-epoch budget. No scheduler, AMP, early stopping, sampler change or prompt search. Batch 8 must still be smoke-checked manually; no GPU training measurement is claimed here.


In [ ]:
PREDECLARE_RUN = False
RUN_DEVELOPMENT_EVALUATION = False
RUN_TRAINING = False
SAVE_CHECKPOINTS = False
EXPORT_RESULTS = False
VERIFY_CHECKPOINT = False
WP3A_PARENT = '300f1d78bb962f67fa2dbdb7b76ed23301abd7df'
CLIP_REVISION = 'd05afc436d78f1c48dc0dbf8e5980a9d471f35f6'
MODEL_REGISTRY = {'ViT-L/14': {'filename': 'ViT-L-14.pt', 'sha256': 'b8cca3fd41ae0c99ba7e8951adf17d267cdb84cd88be6f7c2e0eca1737a03836', 'embedding_dimension': 768, 'input_resolution': 224}}
PROMPT_POLICY = {'version': 'label_only_single_template_v2', 'normalization': 'Unicode NFKC; underscores to spaces; collapse whitespace; lowercase; no aliases/symptoms', 'healthy': 'a photo of a healthy {crop} plant', 'diseased': 'a photo of {article} {crop} plant affected by {disease}', 'article_rule': 'an for normalized crop labels starting with a/e/i/o/u; otherwise a; diseased prompts only; verified for the current 55 crop labels'}
COLUMN_NAMES = ['image_name', 'crop_id', 'disease_id', 'composition_id', 'crop_name', 'disease_name', 'composition_name']
SPLIT_VERSION = 'pytorch-v1'
DEFAULT_CSVS = ('DPD_Small_train.csv',)
EXPECTED_IDENTITIES = {'source_sha256': '9f1b729ab08f120c9e8d3a5681aa1ccbdf1d9210f9aaa6143959363dcb106a7d', 'candidate_sha256': '4cb0a24a96cf647f32311817fa7b082a46d3cf00c20ae61f4287e83cc898dfaf', 'split_sha256': 'f0c7ba212449fd68f1d19c0f84cc91bf4cc1af29ae405128e5198bce89f25f85'}
EXPECTED_DIRECT_PROMPT_SHA256 = '753672c7964bcb20a1346145f21a72b40a19f55590a006b5783823f23c13f537'
ADAPTER_SPEC = {'id': 'bounded_residual_visual_768_64_v1', 'feature_dim': 768, 'bottleneck': 64, 'residual_scale': 0.1, 'activation': 'ReLU', 'bias': True, 'up_initialization': 'zeros', 'parameters': 99136}
PROTOTYPE_SPEC = {'source': 'only the 258 fitting-composition text embeddings; 323 direct candidates retained for evaluation', 'aggregation': 'unit float64 text vectors; equal-candidate mean per original component ID; unit float64 then unit float32', 'image_features_or_frequency_weights_used': False, 'trainable': False}
LOSS_SPEC = {'crop_weight': 0.5, 'disease_weight': 0.5, 'identity_weight': 1.0, 'temperature': 0.07, 'identity': 'mean sample squared L2 between unit adapted and unit frozen features'}
NATIVE_PREPROCESSING = {'resize_shorter_edge': 224, 'interpolation': 'bicubic', 'center_crop': 224, 'convert': 'RGB', 'mean': [0.48145466, 0.4578275, 0.40821073], 'std': [0.26862954, 0.26130258, 0.27577711], 'augmentation': None}
SELECTION_POLICY = {'primary': 'maximum development H@1', 'secondary': 'minimum sample-weighted combined validation objective', 'tertiary': 'earliest epoch', 'full_precision': True, 'epoch_zero_eligible': True}
CANONICAL_PROJECT_ROOT = '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML Project'
WORKSPACE_DIRECTORY = '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML Project/workspace'
HANDOFF_DIRECTORY = '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML Project/workspace/handoffs'
CONFIG = {'model': 'ViT-L/14', 'seed': 42, 'batch_size': 8, 'text_batch_size': 32, 'num_workers': 0, 'device': 'cuda', 'learning_rate': 0.0001, 'weight_decay': 0.0001, 'epochs': 5, 'conditional_weight': 1.0, 'residual_cap': 0.2, 'feature_cache_path': None, 'feature_cache_sha256': None, 'weight_directory': '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML Project/workspace/pretrained_weights/clip', 'image_directory': '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML Project/workspace/datasets/DPD_Small_dataset', 'output_directory': '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML Project/workspace/experiments/wp3b-m4'}
RUN_DIRECTORY = None
FROZEN_CLIP = None
PREPROCESS = None
MODEL_PROVENANCE = None
TEXT_STATE = None
TEXT_PROVENANCE = None
ADAPTER = None
BASELINE_RECORD = None
BASELINE_VALIDATION = None
BASELINE_CHECKPOINT = None
PILOT_RESULTS = None
TRAINING_STARTED = False

RUN_RECORD_DIAGNOSTICS = False
RUN_COMPARISON_REPORT = False
M3_SPEC = {"id": "same_crop_fitting_contrast_v1", "temperature": 0.07, "conditional_weight": [1.0], "negative_pool": "258 fitting-composition direct text vectors; same true crop only", "singleton": "zero loss; mean over all processed fitting samples", "validation_loss": "unchanged M2 component+identity objective", "inference": "unchanged 323 direct float64 cosine scores"}
CONTROL_SUMMARY_PATH = None
INTERVENTION_SUMMARY_PATH = None

BUILD_FEATURE_CACHE = False
LOAD_FEATURE_CACHE = False
FEATURE_CACHE = None
RESIDUAL_CAP_SPEC = {"choices": [None, 0.2], "relative_norm_bound": "norm(y-x)/norm(x) <= rho", "max_angle_degrees": 11.536959032815489, "control": "unbounded M3 adapter", "loss": "unchanged M3 conditional objective"}

RUN_PAIRED_PILOT = False
PAIRED_RESULTS = None
PAIRED_STARTED = False


## 3. Visible training-only source loading and accepted split
The original 8,000 rows produce fitting 4,868 / seen validation 1,238 / internally unseen validation 1,894. Original component/global composition IDs stay unchanged. This loader only accepts the fixed training-source filename; no official-test metadata is used.


In [ ]:
def load_metadata(path):
    if Path(path).name != "DPD_Small_train.csv" or Path(path).resolve().name != "DPD_Small_train.csv":
        raise ValueError("M2 only reads training-source metadata.")
    frame = pd.read_csv(path, header=None)
    if frame.shape[1] != len(COLUMN_NAMES) or frame.empty:
        raise ValueError(f"Expected non-empty seven-column metadata: {path}")
    frame.columns = COLUMN_NAMES
    if frame.isna().any().any():
        raise ValueError(f"Missing metadata values: {path}")
    for col in ("crop_id", "disease_id", "composition_id"):
        numeric = pd.to_numeric(frame[col], errors="raise")
        if ((numeric < 0) | (numeric % 1 != 0)).any():
            raise ValueError(f"Invalid {col}: {path}")
        frame[col] = numeric.astype("int64")
    for name in frame.image_name:
        if not isinstance(name, str) or name in (".", "..") or any(c in name for c in ("/", "\\", ":")):
            raise ValueError(f"Expected flat image filename, got {name!r}")
    return frame


def validate_metadata(train, seen, unseen):
    frames = [train, seen, unseen]
    for frame in frames:
        if frame.image_name.duplicated().any():
            raise ValueError("Duplicate image names within a split")
    for i in range(3):
        for j in range(i + 1, 3):
            if set(frames[i].image_name) & set(frames[j].image_name):
                raise ValueError("Image filename overlap across official splits")
    all_rows = pd.concat(frames)
    for kind in ("crop", "disease", "composition"):
        pairs = all_rows[[f"{kind}_id", f"{kind}_name"]].drop_duplicates()
        if pairs[f"{kind}_id"].duplicated().any() or pairs[f"{kind}_name"].duplicated().any():
            raise ValueError(f"Inconsistent {kind} ID/name mapping")
    pairs = all_rows[["composition_id", "crop_id", "disease_id"]].drop_duplicates()
    if pairs.composition_id.duplicated().any() or pairs.duplicated(["crop_id", "disease_id"]).any():
        raise ValueError("Inconsistent composition/component mapping")
    if set(train.composition_id) & set(unseen.composition_id):
        raise ValueError("Official unseen compositions appear in training")
    if not set(seen.composition_id) <= set(train.composition_id):
        raise ValueError("Seen-test contains compositions absent from training")
    for col in ("crop_id", "disease_id"):
        if set(train[col]) != set(range(train[col].nunique())):
            raise ValueError(f"Training {col} must be contiguous and start at zero")
        if not set(all_rows[col]) <= set(train[col]):
            raise ValueError(f"Test {col} absent from training")
    return {"split_sizes": [len(f) for f in frames],
            "num_crops": int(train.crop_id.nunique()),
            "num_diseases": int(train.disease_id.nunique()),
            "num_seen_compositions": int(train.composition_id.nunique()),
            "num_unseen_compositions": int(unseen.composition_id.nunique()),
            "singleton_compositions": int((train.composition_id.value_counts() == 1).sum()),
            "filename_overlap": False, "seen_unseen_composition_overlap": False}


def make_validation_split(frame, val_split=0.2, seed=42, mode="stratified"):
    """Classwise split keeps singletons in train; heldout removes whole classes.

    Classwise rounding makes the realized image fraction approximate.
    heldout mode interprets val_split as a fraction of composition classes.
    """
    if not 0 < val_split < 1:
        raise ValueError("val_split must be between 0 and 1")
    frame = frame.reset_index(drop=True)
    rng = random.Random(seed)
    groups = {int(key): list(group.index) for key, group in frame.groupby("composition_id", sort=True)}
    val_indices = []
    if mode == "stratified":
        for indices in groups.values():
            rng.shuffle(indices)
            n = min(len(indices) - 1, max(1, round(len(indices) * val_split))) if len(indices) > 1 else 0
            val_indices.extend(indices[:n])
    elif mode == "heldout_composition":
        candidates = list(groups)
        rng.shuffle(candidates)
        remaining_crop = Counter(frame.crop_id)
        remaining_disease = Counter(frame.disease_id)
        target = max(1, round(len(groups) * val_split))
        heldout = []
        for comp_id in candidates:
            group = frame.loc[groups[comp_id]]
            crops, diseases = Counter(group.crop_id), Counter(group.disease_id)
            if all(remaining_crop[k] > n for k, n in crops.items()) and all(remaining_disease[k] > n for k, n in diseases.items()):
                heldout.append(comp_id)
                val_indices.extend(groups[comp_id])
                remaining_crop.subtract(crops)
                remaining_disease.subtract(diseases)
                if len(heldout) == target:
                    break
        if len(heldout) < target:
            raise ValueError(f"Only {len(heldout)} valid heldout classes; requested {target}. Reduce val_split.")
    else:
        raise ValueError(f"Unknown split mode: {mode}")
    validation = frame.loc[sorted(val_indices)].reset_index(drop=True)
    training = frame.drop(index=val_indices).reset_index(drop=True)
    if training.empty or validation.empty:
        raise ValueError("Split produced an empty training or validation set")
    for col in ("crop_id", "disease_id"):
        if set(frame[col]) != set(training[col]):
            raise ValueError(f"Split removed a training component: {col}")
    return training, validation


@dataclass(frozen=True)
class DevelopmentSplit:
    """Original labels and one candidate universe for both validation folds.

    Treat the contained DataFrames/mappings as read-only. Frozen field bindings
    do not make pandas objects immutable; DPDataset copies each frame.
    """

    fit: pd.DataFrame
    val_seen: pd.DataFrame
    val_unseen: pd.DataFrame
    mappings: dict[str, Any]
    candidate_ids: tuple[int, ...]
    candidate_pairs: tuple[tuple[int, int], ...]
    source_csv: str
    source_sha256: str
    candidate_sha256: str
    seed: int
    heldout_fraction: float
    seen_fraction: float
    split_version: str = f"{SPLIT_VERSION}/three-way-v1"


def _fraction(value: float, name: str) -> float:
    if isinstance(value, bool) or not isinstance(value, Real) or not 0 < value < 1:
        raise ValueError(f"{name} must be a finite fraction between zero and one")
    return float(value)


def _training_mappings(frame: pd.DataFrame) -> dict[str, Any]:
    """Build only training-derived maps; the combined train/test wrapper is unsuitable."""
    mappings = {}
    for kind in ("crop", "disease"):
        pairs = frame[[f"{kind}_id", f"{kind}_name"]].drop_duplicates()
        pairs = pairs.sort_values(f"{kind}_id")
        id_to_name = {int(key): str(name) for key, name in pairs.itertuples(index=False, name=None)}
        mappings[f"{kind}_id_to_name"] = id_to_name
        mappings[f"{kind}_name_to_id"] = {name: key for key, name in id_to_name.items()}
        mappings[f"num_{kind}s"] = len(id_to_name)
    pairs = frame[["composition_id", "crop_id", "disease_id", "composition_name"]]
    pairs = pairs.drop_duplicates().sort_values("composition_id")
    mappings["composition_id_to_pair"] = {
        int(row.composition_id): (int(row.crop_id), int(row.disease_id))
        for row in pairs.itertuples()
    }
    mappings["composition_id_to_name"] = {
        int(row.composition_id): str(row.composition_name) for row in pairs.itertuples()
    }
    ids = tuple(mappings["composition_id_to_pair"])
    mappings["composition_id_to_index"] = {comp: index for index, comp in enumerate(ids)}
    mappings["composition_index_to_id"] = dict(enumerate(ids))
    mappings["num_compositions"] = len(ids)
    return mappings


def _check_partitions(
    source: pd.DataFrame, fit: pd.DataFrame, seen: pd.DataFrame, unseen: pd.DataFrame,
) -> None:
    combined = pd.concat([fit, seen, unseen], ignore_index=True)
    if (combined.image_name.duplicated().any()
            or set(combined.image_name) != set(source.image_name)):
        raise ValueError("Development partitions must be disjoint and cover training filenames")
    if not source.sort_values("image_name").reset_index(drop=True).equals(
        combined.sort_values("image_name").reset_index(drop=True)
    ):
        raise ValueError("Development partitions changed original metadata rows or label types")
    fit_ids, seen_ids, unseen_ids = (set(frame.composition_id) for frame in (fit, seen, unseen))
    if not seen_ids <= fit_ids or unseen_ids & (fit_ids | seen_ids):
        raise ValueError("Seen coverage or held-out composition isolation failed")
    for column in ("crop_id", "disease_id"):
        if set(fit[column]) != set(source[column]):
            raise ValueError(f"Fitting partition lost training components: {column}")


def build_development_split(
    train_csv: str | Path = DEFAULT_CSVS[0],
    *,
    seed: int = 42,
    heldout_fraction: float = 0.2,
    seen_fraction: float = 0.2,
) -> DevelopmentSplit:
    """Compose existing train-only splits without reading images or writing files.

    The default DPD-Small source yields 4868 fit / 1238 seen / 1894 unseen
    images. heldout_fraction counts compositions; seen_fraction is classwise
    image sampling within the retained portion. Synthetic training CSVs are
    supported for tests. Known official test CSV paths are rejected.
    """
    if isinstance(seed, bool) or not isinstance(seed, Integral) or not 0 <= seed < 2**32:
        raise ValueError("seed must be an integer in [0, 2**32)")
    seed = int(seed)
    heldout_fraction = _fraction(heldout_fraction, "heldout_fraction")
    seen_fraction = _fraction(seen_fraction, "seen_fraction")
    path = Path(train_csv)
    if path.name != "DPD_Small_train.csv" or path.resolve().name != "DPD_Small_train.csv":
        raise ValueError("Official test metadata cannot be used as development training data")
    before_hash = hashlib.sha256(path.read_bytes()).hexdigest()
    source = load_metadata(path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != before_hash:
        raise ValueError("Training CSV changed while loading")
    # Empty control frames reuse WP1 validation without loading either test CSV.
    empty = source.iloc[:0]
    validate_metadata(source, empty, empty)
    retained, unseen = make_validation_split(
        source, val_split=heldout_fraction, seed=seed, mode="heldout_composition"
    )
    fit, seen = make_validation_split(
        retained, val_split=seen_fraction, seed=seed, mode="stratified"
    )
    _check_partitions(source, fit, seen, unseen)
    mappings = _training_mappings(source)
    ids = tuple(mappings["composition_id_to_pair"])
    pairs = tuple(mappings["composition_id_to_pair"][comp] for comp in ids)
    records = [[comp, *pair] for comp, pair in zip(ids, pairs)]
    candidate_hash = hashlib.sha256(
        json.dumps(records, separators=(",", ":")).encode("utf-8")
    ).hexdigest()
    return DevelopmentSplit(
        fit=fit, val_seen=seen, val_unseen=unseen, mappings=mappings,
        candidate_ids=ids, candidate_pairs=pairs, source_csv=str(path.resolve()),
        source_sha256=before_hash, candidate_sha256=candidate_hash, seed=seed,
        heldout_fraction=heldout_fraction, seen_fraction=seen_fraction,
    )


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(4 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def json_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()


def split_identity(split):
    records = {
        name: [[str(row.image_name), int(row.crop_id), int(row.disease_id), int(row.composition_id)]
               for row in frame.itertuples()]
        for name, frame in (("fit", split.fit), ("val_seen", split.val_seen), ("val_unseen", split.val_unseen))
    }
    return {"source_sha256": split.source_sha256, "candidate_sha256": split.candidate_sha256,
            "partition_sha256": {name: json_hash(rows) for name, rows in records.items()},
            "split_sha256": json_hash({name: frame.image_name.tolist() for name, frame in (("fit", split.fit), ("val_seen", split.val_seen), ("val_unseen", split.val_unseen))}), "split_version": split.split_version, "split_seed": split.seed}

## 4. Unchanged frozen CLIP, direct prompts and primary scorer
The approved v2 direct-composition prompts, native tokenizer/transforms, official local artifact/revision checks, float64 cosine scorer and ascending-global-ID exact ties are retained verbatim. All helper definitions precede their first execution.


In [ ]:
def normalize_label(label):
    if not isinstance(label, str):
        raise TypeError("Labels must be strings.")
    cleaned = " ".join(unicodedata.normalize("NFKC", label).replace("_", " ").split()).lower()
    if not cleaned:
        raise ValueError("An empty label cannot form a prompt.")
    return cleaned


def crop_indefinite_article(crop_label):
    """Label-only rule, checked against the current 55-crop vocabulary."""
    crop = normalize_label(crop_label)
    return "an" if crop[0] in "aeiou" else "a"


def build_prompt_table(split):
    rows = []
    for column, (global_id, pair) in enumerate(zip(split.candidate_ids, split.candidate_pairs)):
        crop_id, disease_id = pair
        crop = normalize_label(split.mappings["crop_id_to_name"][crop_id])
        disease = normalize_label(split.mappings["disease_id_to_name"][disease_id])
        prompt = (PROMPT_POLICY["healthy"].format(crop=crop) if disease == "healthy"
                  else PROMPT_POLICY["diseased"].format(article=crop_indefinite_article(crop), crop=crop, disease=disease))
        rows.append({"score_column": column, "global_composition_id": global_id,
                     "crop_id": crop_id, "disease_id": disease_id, "crop": crop, "disease": disease,
                     "prompt": prompt})
    table = pd.DataFrame(rows)
    if len(table) != len(split.candidate_ids) or table.empty:
        raise ValueError("Candidate/prompt lengths disagree.")
    ids = table.global_composition_id.tolist()
    if ids != sorted(set(ids)) or list(zip(table.crop_id, table.disease_id)) != list(split.candidate_pairs):
        raise ValueError("Candidate IDs must be unique ascending global IDs with exact component alignment.")
    if table.prompt.duplicated().any():
        raise ValueError("Distinct candidates normalize to duplicate prompts; review the metadata.")
    return table


def unit_features(features):
    if not isinstance(features, torch.Tensor) or features.ndim != 2 or not features.is_floating_point():
        raise ValueError("Features must be a floating [N,D] tensor.")
    values = features.detach().to(device="cpu", dtype=torch.float64)
    if values.shape[0] == 0 or values.shape[1] == 0 or not torch.isfinite(values).all():
        raise ValueError("Feature vectors must be nonempty, finite and nonzero.")
    scale = values.abs().amax(dim=1, keepdim=True)
    if (scale == 0).any():
        raise ValueError("Zero feature vectors cannot be normalized.")
    scaled = values / scale
    return scaled / torch.linalg.vector_norm(scaled, dim=1, keepdim=True)


def cosine_candidate_scores(image_features, text_features):
    image = unit_features(image_features)
    text = unit_features(text_features)
    if image.shape[1] != text.shape[1]:
        raise ValueError("Image/text embedding dimensions differ.")
    scores = image @ text.T
    if not torch.isfinite(scores).all():
        raise ValueError("Non-finite cosine similarity.")
    return scores


def rank_global_ids(scores, candidate_ids):
    if not isinstance(scores, torch.Tensor) or scores.ndim != 2 or not scores.is_floating_point():
        raise ValueError("Scores must be a floating [N,C] tensor.")
    ids = tuple(candidate_ids)
    if not ids or any(type(i) is not int or i < 0 for i in ids) or list(ids) != sorted(set(ids)):
        raise ValueError("Candidates must be unique ascending nonnegative global IDs.")
    if scores.shape[1] != len(ids) or scores.shape[0] == 0 or not torch.isfinite(scores).all():
        raise ValueError("Score shape/values violate the candidate contract.")
    order = torch.argsort(scores.detach().to("cpu", torch.float64), dim=1, descending=True, stable=True)
    return torch.tensor(ids, dtype=torch.int64)[order]


def topk_accuracy(ranked_global_ids, truths, k):
    truths = torch.as_tensor(truths)
    if (ranked_global_ids.ndim != 2 or ranked_global_ids.dtype != torch.int64 or
            truths.ndim != 1 or truths.dtype != torch.int64 or len(truths) != len(ranked_global_ids)
            or len(truths) == 0):
        raise ValueError("Rankings/truths must be nonempty aligned int64 global IDs.")
    if type(k) is not int or not 1 <= k <= ranked_global_ids.shape[1]:
        raise ValueError("k must lie inside the candidate count.")
    if not (ranked_global_ids == truths.cpu()[:, None]).any(dim=1).all():
        raise ValueError("A composition truth is absent from the candidate universe.")
    correct = (ranked_global_ids[:, :k] == truths.cpu()[:, None]).any(dim=1)
    return correct.to(torch.float64).mean().item()


def harmonic_mean(seen, unseen):
    for value in (seen, unseen):
        if isinstance(value, bool) or not isinstance(value, Real) or not math.isfinite(value) or not 0 <= value <= 1:
            raise ValueError("Accuracies must be finite fractions in [0,1].")
    return 0.0 if seen + unseen == 0 else 2.0 * seen * unseen / (seen + unseen)


def fold_metrics(scores, truths, candidate_ids):
    ranked = rank_global_ids(scores, candidate_ids)
    return {"sample_count": len(ranked), "top1": topk_accuracy(ranked, truths, 1),
            "top5": topk_accuracy(ranked, truths, 5)}, ranked


def weight_spec(model_name):
    spec = dict(MODEL_REGISTRY[model_name])
    spec["url"] = f"https://openaipublic.azureedge.net/clip/models/{spec['sha256']}/{spec['filename']}"
    return spec


def verify_weight(path, expected_sha256):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Missing local CLIP weights: {path}. Explicitly acquire or select the verified file.")
    actual = sha256_file(path)
    if actual != expected_sha256:
        raise ValueError(f"CLIP weight SHA-256 mismatch: {path}")
    return {"path": str(path.resolve()), "sha256": actual, "bytes": path.stat().st_size}


def load_frozen_clip(config):
    import clip
    distribution = metadata.distribution("clip")
    direct = json.loads(distribution.read_text("direct_url.json") or "{}")
    if distribution.version != "1.0" or direct.get("vcs_info", {}).get("commit_id") != CLIP_REVISION:
        raise RuntimeError("Install the approved official CLIP revision; this loader refuses an ambiguous package.")
    spec = weight_spec(config["model"])
    artifact = verify_weight(Path(config["weight_directory"]) / spec["filename"], spec["sha256"])
    device = torch.device(config["device"])
    # A fully hash-verified official local artifact prevents clip.load's named-model downloader.
    model, preprocess = clip.load(artifact["path"], device=device, jit=False)
    model.eval().requires_grad_(False)
    if int(model.visual.input_resolution) != spec["input_resolution"]:
        raise ValueError("Unexpected CLIP image resolution.")
    return model, preprocess, {"backend": "official_openai_clip", "revision": CLIP_REVISION,
        "model": config["model"], "artifact": artifact, "official_url": spec["url"],
        "native_preprocess": repr(preprocess), "context_length": int(model.context_length),
        "tokenizer": "clip.tokenize(context_length=77, truncate=False)",
        "bpe_sha256": sha256_file(Path(clip.__file__).parent / "bpe_simple_vocab_16e6.txt.gz"),
        "parameters": sum(p.numel() for p in model.parameters()),
        "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
        "logit_scale": model.logit_scale.detach().float().exp().cpu().item(),
        "first_parameter_dtype": str(next(model.parameters()).dtype),
        "image_encoder_dtype": str(getattr(model, "dtype", next(model.parameters()).dtype))}


def tokenize_candidates(table):
    import clip
    if table.empty or table.prompt.duplicated().any():
        raise ValueError("Prompts must be nonempty and unique.")
    tokens = clip.tokenize(table.prompt.tolist(), context_length=77, truncate=False)
    if tokens.shape != (len(table), 77):
        raise ValueError("Native tokenizer returned an unexpected shape.")
    return tokens


@torch.inference_mode()
def encode_candidate_texts(model, table, config):
    if model.training or any(p.requires_grad for p in model.parameters()):
        raise ValueError("CLIP must be frozen and in evaluation mode.")
    tokens = tokenize_candidates(table)
    outputs = []
    device = torch.device(config["device"])
    for start in range(0, len(tokens), config["text_batch_size"]):
        outputs.append(model.encode_text(tokens[start:start + config["text_batch_size"]].to(device)).cpu())
    features = unit_features(torch.cat(outputs))
    if features.shape != (len(table), MODEL_REGISTRY[config["model"]]["embedding_dimension"]):
        raise ValueError("Unexpected CLIP text feature contract.")
    return features, {"tokens_shape": list(tokens.shape), "features_shape": list(features.shape),
                      "tokens_sha256": hashlib.sha256(tokens.numpy().tobytes()).hexdigest(),
                      "prompt_sha256": json_hash({"policy": PROMPT_POLICY, "rows": table.to_dict("records")})}


class CLIPDevelopmentDataset(Dataset):
    def __init__(self, frame, image_directory, native_preprocess, allowed_training_names):
        if native_preprocess is None:
            raise ValueError("Supply the selected CLIP model's native preprocessing.")
        if frame.empty or frame.image_name.duplicated().any():
            raise ValueError("A nonempty unique-filename development frame is required.")
        if not set(frame.image_name) <= set(allowed_training_names):
            raise ValueError("Dataset contains filenames outside original training metadata.")
        self.frame = frame.copy().reset_index(drop=True)
        self.image_directory = Path(image_directory).resolve()
        self.preprocess = native_preprocess
        for name in self.frame.image_name:
            if not isinstance(name, str) or name in (".", "..") or any(c in name for c in ("/", "\\", ":")):
                raise ValueError("Only flat training image filenames are allowed.")
            path = (self.image_directory / name).resolve()
            if path.parent != self.image_directory:
                raise ValueError("Image path escapes the approved directory.")

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        path = self.image_directory / row.image_name
        with Image.open(path) as image:
            image.load()  # Explicit decoding while the file is open.
            tensor = self.preprocess(image.convert("RGB"))
        if tensor.ndim != 3 or not torch.isfinite(tensor).all():
            raise ValueError(f"Invalid native-preprocessed image: {row.image_name}")
        return tensor, {
            "crop": int(row.crop_id), "disease": int(row.disease_id),
            "composition": int(row.composition_id), "image_name": str(row.image_name),
        }

## 5. Adapter and one fixed component-aware objective
For frozen raw image feature x, let u = x / ||x||. The adapter returns:
z_raw = x + 0.1 ||x|| W_up ReLU(W_down u + b_down) + corresponding scaled up bias.

Dimensions are 768→64→768, with 99,136 trainable parameters. W_up and b_up start at zero, giving exact identity on raw half/float32 CLIP features. Raw scale is preserved; the unchanged primary scorer performs its usual float64 normalization afterward.

Crop and disease anchors are equal-candidate averages of only the 258 fitting-composition text vectors grouped by original component ID, then normalized. They use semantic labels from the approved candidate universe, without validation images or fitting frequency weights.

L = 0.5 CE(crop cosine logits / 0.07) + 0.5 CE(disease cosine logits / 0.07)
    + 1.0 mean_i ||unit(z_raw_i) - unit(x_i)||².

There is no composition-class CE that treats held-out pairs as never-positive supervised classes. Primary evaluation remains direct composition cosine over all 323 candidates. Components in the objective are fixed contextual anchors, not trainable heads.

At the first backward pass, the zero up-projection blocks data gradients to W_down; W_up receives a gradient. Later steps propagate to W_down. AdamW decay may still act on a zero-gradient parameter.


In [ ]:
class ResidualVisualAdapter(torch.nn.Module):
    """A 99,136-parameter visual residual; exact raw-feature identity at initialization."""
    def __init__(self, residual_cap=None):
        super().__init__()
        if residual_cap is not None and (type(residual_cap) not in (int, float) or residual_cap != .2):
            raise ValueError("Only the predeclared unbounded control or rho=0.20 is supported.")
        self.residual_cap = residual_cap
        self.down = torch.nn.Linear(768, 64)
        self.up = torch.nn.Linear(64, 768)
        torch.nn.init.kaiming_uniform_(self.down.weight, a=math.sqrt(5))
        torch.nn.init.zeros_(self.down.bias)
        torch.nn.init.zeros_(self.up.weight)
        torch.nn.init.zeros_(self.up.bias)

    def forward(self, features):
        if not isinstance(features, torch.Tensor) or features.ndim != 2 or features.shape[1] != 768:
            raise ValueError("Expected frozen CLIP image embeddings [B,768].")
        base = features.detach().float()
        norms = torch.linalg.vector_norm(base, dim=1, keepdim=True)
        if (not len(base) or not torch.isfinite(base).all()
                or not torch.isfinite(norms).all() or (norms <= 0).any()):
            raise ValueError("Image features must be finite and nonzero.")
        unit = base / norms
        residual = self.up(torch.relu(self.down(unit)))
        if not torch.isfinite(residual).all():
            raise ValueError("Non-finite learned adapter residual.")
        # Preserve the accepted arithmetic for every finite valid input in both arms.
        if self.residual_cap is None:
            output = base + 0.1 * norms * residual  # Exact M3 control arithmetic.
        else:
            delta = 0.1 * residual
            size = torch.linalg.vector_norm(delta, dim=1, keepdim=True)
            if not torch.isfinite(size).all():
                raise ValueError("Non-finite relative residual size; no norm guarantee is possible.")
            factor = (self.residual_cap / size.clamp_min(torch.finfo(delta.dtype).eps)).clamp(max=1.0)
            output = base + norms * (delta * factor)
        output_norms = torch.linalg.vector_norm(output, dim=1)
        if (not torch.isfinite(output).all() or not torch.isfinite(output_norms).all()
                or (output_norms <= 0).any()):
            raise ValueError("Adapter output must be finite and nonzero.")
        return output


def require_frozen_clip(model):
    if model.training or any(p.requires_grad or p.grad is not None for p in model.parameters()):
        raise ValueError("CLIP must be eval-mode, frozen, and have no accumulated gradients.")


@torch.no_grad()
def component_prototypes(direct_features, table, fitting_candidate_ids=None):
    vectors = unit_features(direct_features)
    if len(vectors) != len(table):
        raise ValueError("Direct text rows and candidate table differ.")
    if fitting_candidate_ids is None:
        raise ValueError("Explicit fitting-only candidate IDs are required.")
    pool = list(fitting_candidate_ids)
    if not pool or pool != sorted(set(pool)) or not set(pool) <= set(table.global_composition_id):
        raise ValueError("Invalid fitting-only prototype pool.")
    mask = table.global_composition_id.isin(pool).to_numpy()
    vectors = vectors[mask]
    table = table.loc[mask].reset_index(drop=True)
    result = {}
    for kind in ("crop", "disease"):
        ids = tuple(int(x) for x in sorted(table[kind + "_id"].unique()))
        means = torch.stack([vectors[table[kind + "_id"].to_numpy() == key].mean(0) for key in ids])
        # Fixed fitting-composition text averages; no image features or frequency weights.
        anchors = unit_features(means).float().clone()
        result[kind] = torch.nn.functional.normalize(anchors, dim=1).detach()
        result[kind + "_ids"] = ids
    return result


def adapter_objective(adapted, original, crop_labels, disease_labels, prototypes):
    if (adapted.ndim != 2 or adapted.shape != original.shape or adapted.shape[1] != 768
            or not len(adapted) or not torch.isfinite(adapted).all()
            or not torch.isfinite(original).all()
            or (torch.linalg.vector_norm(original.detach().float(), dim=1) <= 0).any()
            or (torch.linalg.vector_norm(adapted.float(), dim=1) <= 0).any()):
        raise ValueError("Loss inputs must be finite aligned [B,768] embeddings.")
    y = torch.nn.functional.normalize(adapted.float(), dim=1)
    x = torch.nn.functional.normalize(original.detach().float(), dim=1)
    losses = {}
    for kind, labels in (("crop", crop_labels), ("disease", disease_labels)):
        anchor = prototypes[kind]
        ids = tuple(prototypes[kind + "_ids"])
        if (anchor.ndim != 2 or anchor.shape != (len(ids), 768) or anchor.requires_grad
                or not torch.isfinite(anchor).all() or len(set(ids)) != len(ids)):
            raise ValueError("Component anchors must be fixed, finite and correctly mapped.")
        if (not isinstance(labels, torch.Tensor) or labels.dtype != torch.int64
                or labels.shape != (len(y),)):
            raise ValueError("Labels must be aligned original int64 component IDs.")
        columns = {key: col for col, key in enumerate(ids)}
        values = labels.detach().cpu().tolist()
        if not set(values) <= set(columns):
            raise ValueError("Component label is absent from the fixed prototype map.")
        target = torch.tensor([columns[v] for v in values], dtype=torch.int64, device=y.device)
        logits = y @ anchor.to(device=y.device, dtype=torch.float32).T / 0.07
        losses[kind + "_loss"] = torch.nn.functional.cross_entropy(logits, target)
    identity = ((y - x) ** 2).sum(dim=1).mean()
    total = .5 * losses["crop_loss"] + .5 * losses["disease_loss"] + 1.0 * identity
    if not torch.isfinite(total):
        raise ValueError("Non-finite adapter objective.")
    return {"loss": total, **losses, "identity_loss": identity}


def make_adapter_optimizer(adapter, config):
    params = list(adapter.parameters())
    if len(set(id(p) for p in params)) != len(params) or not all(p.requires_grad for p in params):
        raise ValueError("Expected distinct trainable adapter parameters only.")
    return torch.optim.AdamW(params, lr=config["learning_rate"], weight_decay=config["weight_decay"])


def tensor_sha256(tensor):
    return hashlib.sha256(tensor.detach().cpu().contiguous().numpy().tobytes()).hexdigest()


def seal_text_state(direct, prototypes, contract):
    # Clone outside inference_mode so fixed anchors can be saved for adapter backward.
    state = {"direct": direct.detach().cpu().clone(),
             "crop": prototypes["crop"].detach().cpu().float().clone(),
             "disease": prototypes["disease"].detach().cpu().float().clone(),
             "crop_ids": list(prototypes["crop_ids"]), "disease_ids": list(prototypes["disease_ids"]),
             "candidate_ids": list(contract["candidate_ids"]),
             "prototype_pool_ids": list(contract["fitting_composition_ids"]), "contract_sha256": json_hash(contract)}
    state["hashes"] = {key: tensor_sha256(state[key]) for key in ("direct", "crop", "disease")}
    state.update(build_conditional_state(state["direct"], contract))
    validate_text_state(state, contract)
    return state


def validate_text_state(state, contract):
    if state["contract_sha256"] != json_hash(contract):
        raise ValueError("Text state belongs to a different predeclared experiment.")
    if state["candidate_ids"] != contract["candidate_ids"]:
        raise ValueError("Reordered global candidate mapping.")
    if state["prototype_pool_ids"] != contract["fitting_composition_ids"]:
        raise ValueError("Prototype training-pool provenance changed.")
    for kind, count in (("direct", 323), ("crop", 55), ("disease", 175)):
        tensor = state[kind]
        if (tensor.shape != (count, 768) or tensor.requires_grad
                or tensor.dtype != (torch.float64 if kind == "direct" else torch.float32)
                or not torch.isfinite(tensor).all()
                or tensor_sha256(tensor) != state["hashes"][kind]):
            raise ValueError("Text/prototype bank changed: " + kind)
    for kind in ("crop", "disease"):
        if list(state[kind + "_ids"]) != contract["component_ids"][kind]:
            raise ValueError("Reordered component mapping.")
    validate_conditional_state(state, contract)
    return True


def prepare_text_state(model, table, config, contract):
    require_frozen_clip(model)
    # Exact accepted WP3A tokenization, text batches and float64 normalization.
    direct, provenance = encode_candidate_texts(model, table, config)
    prototypes = component_prototypes(direct, table, contract["fitting_composition_ids"])
    return seal_text_state(direct, prototypes, contract), provenance


class M2ImageDataset(CLIPDevelopmentDataset):
    def __init__(self, split, role, image_directory, native_preprocess):
        if role not in ("fit", "val_seen", "val_unseen"):
            raise ValueError("Only training-derived folds are supported.")
        frame = getattr(split, role)
        super().__init__(frame, image_directory, native_preprocess, set(frame.image_name))
        self.role = role


def make_m4_loader(split, role, preprocess, config):
    verify_split_identity(split)
    if FEATURE_CACHE is None:
        raise ValueError("Explicitly load and validate the shared feature cache first.")
    dataset = CachedFeatureDataset(split, role, FEATURE_CACHE)
    return DataLoader(dataset, batch_size=config["batch_size"], shuffle=role == "fit", drop_last=False,
        num_workers=0, pin_memory=torch.device(config["device"]).type == "cuda",
        generator=torch.Generator().manual_seed(config["seed"]))


def check_batch_targets(targets, expected, seen_names):
    names = list(targets["image_name"])
    if not names or len(set(names)) != len(names) or set(names) & seen_names:
        raise ValueError("Empty or repeated image identities.")
    if not set(names) <= set(expected.index):
        raise ValueError("Batch contains a filename outside the approved partition.")
    rows = expected.loc[names]
    for key, column in (("crop", "crop_id"), ("disease", "disease_id"), ("composition", "composition_id")):
        labels = torch.as_tensor(targets[key])
        if labels.dtype != torch.int64 or labels.shape != (len(names),):
            raise ValueError("Original int64 label compatibility failed.")
        if labels.cpu().tolist() != rows[column].tolist():
            raise ValueError("Batch labels no longer match the source filename.")
    seen_names.update(names)
    return names


## Retained M3 fitting-only conditional objective
Both M4 arms keep L_M2 + L_same_crop with conditional_weight=1.0. The 258 fitting-composition text vectors, healthy negatives where supported and singleton zero contribution are unchanged. Validation keeps the base M2 objective for the exact loss tie-break. Only the residual cap differs between arms.


In [ ]:
def build_conditional_state(direct, contract):
    ids = list(contract["candidate_ids"])
    pool = list(contract["fitting_composition_ids"])
    if pool != sorted(set(pool)) or not pool or not set(pool) <= set(ids):
        raise ValueError("Explicit ordered fitting-only conditional pool required.")
    positions = [ids.index(i) for i in pool]
    pairs = [contract["candidate_pairs"][i] for i in positions]
    text = torch.nn.functional.normalize(direct[positions].detach().cpu().float().clone(), dim=1)
    return {"conditional_ids": torch.tensor(pool, dtype=torch.int64),
            "conditional_pairs": torch.tensor(pairs, dtype=torch.int64),
            "conditional_text": text, "conditional_sha256": tensor_sha256(text),
            "conditional_weight": float(contract["hyperparameters"]["conditional_weight"])}


def validate_conditional_state(state, contract):
    expected = build_conditional_state(state["direct"], contract)
    for key in ("conditional_ids", "conditional_pairs", "conditional_text"):
        actual = state[key]
        if (not isinstance(actual, torch.Tensor) or actual.requires_grad
                or actual.dtype != expected[key].dtype or not torch.equal(actual.cpu(), expected[key])):
            raise ValueError("Changed fitting-only conditional bank: " + key)
    if (state["conditional_sha256"] != tensor_sha256(state["conditional_text"])
            or state["conditional_weight"] != expected["conditional_weight"]):
        raise ValueError("Conditional objective/hash mismatch.")


def same_crop_loss(adapted, crop_labels, disease_labels, composition_labels, state):
    """CE among fitting descriptions of the true crop; never used for inference."""
    if (adapted.ndim != 2 or adapted.shape[1] != 768 or not len(adapted)
            or not torch.isfinite(adapted).all()
            or not torch.isfinite(torch.linalg.vector_norm(adapted.float(), dim=1)).all()
            or (torch.linalg.vector_norm(adapted.float(), dim=1) <= 0).any()):
        raise ValueError("Conditional loss expects finite nonzero [B,768] features.")
    labels = (crop_labels, disease_labels, composition_labels)
    if any(not isinstance(y, torch.Tensor) or y.dtype != torch.int64 or y.shape != (len(adapted),) for y in labels):
        raise ValueError("Aligned original int64 crop/disease/composition labels required.")
    ids, pairs, bank = (state[k] for k in ("conditional_ids", "conditional_pairs", "conditional_text"))
    if (ids.ndim != 1 or ids.dtype != torch.int64 or not len(ids)
            or pairs.shape != (len(ids), 2) or pairs.dtype != torch.int64
            or bank.shape != (len(ids), 768) or bank.requires_grad
            or not torch.isfinite(bank).all() or len(set(ids.tolist())) != len(ids)):
        raise ValueError("Malformed frozen conditional bank.")
    lookup = {int(key): j for j, key in enumerate(ids.tolist())}
    truth = composition_labels.detach().cpu().tolist()
    if not set(truth) <= set(lookup):
        raise ValueError("Held-out/unknown compositions cannot enter fitting supervision.")
    columns = torch.tensor([lookup[key] for key in truth], dtype=torch.int64)
    expected = pairs.cpu()[columns]
    if (not torch.equal(expected[:, 0], crop_labels.detach().cpu())
            or not torch.equal(expected[:, 1], disease_labels.detach().cpu())):
        raise ValueError("Composition/component labels disagree.")
    y = torch.nn.functional.normalize(adapted.float(), dim=1)
    logits = y @ bank.to(y.device).T / .07
    mask = pairs[:, 0].to(y.device)[None, :] == crop_labels.to(y.device)[:, None]
    if not mask.any(dim=1).all():
        raise ValueError("No fitting descriptions for a training crop.")
    # -inf excludes other crops exactly; CE/logsumexp stays finite because each row has its target.
    loss = torch.nn.functional.cross_entropy(logits.masked_fill(~mask, float("-inf")), columns.to(y.device))
    if not torch.isfinite(loss):
        raise ValueError("Non-finite conditional objective.")
    return loss


def m3_training_objective(adapted, original, crop_labels, disease_labels, composition_labels, state, weight):
    if type(weight) not in (int, float) or weight not in (0., 1.):
        raise ValueError("Only predeclared control/intervention weights 0 or 1 are supported.")
    base = adapter_objective(adapted, original, crop_labels, disease_labels, state)
    conditional = same_crop_loss(adapted, crop_labels, disease_labels, composition_labels, state)
    total = base["loss"] if weight == 0 else base["loss"] + conditional
    if not torch.isfinite(total):
        raise ValueError("Non-finite M3 training objective.")
    return {**base, "loss": total, "conditional_loss": conditional}


## 6. Explicit fitting optimization and gradient boundary
Only a fitting-role loader is accepted. Every filename and all original labels are checked against the approved fitting frame before each optimizer step. CLIP remains eval/frozen; no_grad extraction supplies constants for adapter backward. Gradients are checked and clipped only on the adapter. Losses use processed sample weighting.


In [ ]:
def train_adapter_epoch(model, adapter, loader, optimizer, state, fit_frame, device, progress=True):
    from tqdm.auto import tqdm
    require_frozen_clip(model)
    if getattr(loader.dataset, "role", None) != "fit" or not len(loader) or fit_frame.empty:
        raise ValueError("Optimization requires a nonempty fitting-only loader.")
    owned = {id(p) for group in optimizer.param_groups for p in group["params"]}
    if owned != {id(p) for p in adapter.parameters()}:
        raise ValueError("Optimizer must contain adapter parameters only.")
    adapter.train()
    expected = fit_frame.set_index("image_name", drop=False)
    seen_names, count = set(), 0
    totals = {k: 0. for k in ("loss", "crop_loss", "disease_loss", "identity_loss", "conditional_loss")}
    device = torch.device(device)
    if device.type == "cuda":
        torch.cuda.synchronize(device)
        torch.cuda.reset_peak_memory_stats(device)
    started = time.perf_counter()
    for images, targets in tqdm(loader, desc="Fitting adapter only", disable=not progress):
        names = check_batch_targets(targets, expected, seen_names)
        if len(images) != len(names) or not torch.isfinite(images).all():
            raise ValueError("Invalid fitting image batch.")
        optimizer.zero_grad(set_to_none=True)
        # no_grad rather than inference_mode: these constants are saved for adapter backward.
        with torch.no_grad():
            original = model.encode_image(images.to(device)).detach().float()
        adapted = adapter(original)
        losses = m3_training_objective(adapted, original, targets["crop"], targets["disease"],
                                       targets["composition"], state, state["conditional_weight"])
        losses["loss"].backward()
        gradients = [p.grad for p in adapter.parameters()]
        if any(g is None or not torch.isfinite(g).all() for g in gradients):
            raise ValueError("Missing or non-finite adapter gradient.")
        torch.nn.utils.clip_grad_norm_(adapter.parameters(), 1.0, error_if_nonfinite=True)
        optimizer.step()
        require_frozen_clip(model)
        n = len(names)
        for key in totals:
            totals[key] += float(losses[key].detach()) * n
        count += n
    if seen_names != set(expected.index) or count != len(fit_frame):
        raise ValueError("Each fitting image must be processed exactly once.")
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    return {**{k: v / count for k, v in totals.items()}, "sample_count": count,
            "elapsed_seconds": time.perf_counter() - started,
            "peak_allocated_bytes": torch.cuda.max_memory_allocated(device) if device.type == "cuda" else None,
            "peak_reserved_bytes": torch.cuda.max_memory_reserved(device) if device.type == "cuda" else None}


@torch.inference_mode()
def validate_adapter(model, adapter, loader, state, expected_frame, candidate_ids, device, progress=True):
    from tqdm.auto import tqdm
    require_frozen_clip(model)
    if getattr(loader.dataset, "role", None) not in ("val_seen", "val_unseen") or not len(loader):
        raise ValueError("Only nonempty development validation loaders are accepted.")
    adapter.eval()
    if list(candidate_ids) != state["candidate_ids"]:
        raise ValueError("Validation candidate ordering mismatch.")
    device = torch.device(device)
    expected = expected_frame.set_index("image_name", drop=False)
    seen_names, filenames, truths, ranks, controls = set(), [], [], [], []
    totals = {k: 0. for k in ("loss", "crop_loss", "disease_loss", "identity_loss")}
    count = 0
    if device.type == "cuda":
        torch.cuda.synchronize(device)
        torch.cuda.reset_peak_memory_stats(device)
    started = time.perf_counter()
    for images, targets in tqdm(loader, desc=loader.dataset.role, disable=not progress):
        names = check_batch_targets(targets, expected, seen_names)
        if len(images) != len(names) or not torch.isfinite(images).all():
            raise ValueError("Invalid validation batch.")
        original = model.encode_image(images.to(device)).float()
        adapted = adapter(original)
        losses = adapter_objective(adapted, original, targets["crop"], targets["disease"], state)
        # Unchanged primary scorer: raw adapted features against the original 323 text vectors.
        scores = cosine_candidate_scores(adapted, state["direct"])
        control = cosine_candidate_scores(original, state["direct"])
        ranks.append(rank_global_ids(scores, candidate_ids))
        controls.append(rank_global_ids(control, candidate_ids))
        truths.append(targets["composition"].cpu())
        filenames.extend(names)
        n = len(names)
        for key in totals:
            totals[key] += float(losses[key]) * n
        count += n
    if filenames != expected_frame.image_name.tolist() or count != len(expected_frame):
        raise ValueError("Validation must preserve the full ordered partition exactly.")
    ranking, control_ranking, truth = torch.cat(ranks), torch.cat(controls), torch.cat(truths)
    metrics = {"top1": topk_accuracy(ranking, truth, 1), "top5": topk_accuracy(ranking, truth, 5)}
    control_metrics = {"top1": topk_accuracy(control_ranking, truth, 1),
                       "top5": topk_accuracy(control_ranking, truth, 5)}
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    return {**{k: v / count for k, v in totals.items()}, "sample_count": count,
        "metrics": metrics, "control_metrics": control_metrics, "ranking": ranking,
        "control_ranking": control_ranking, "truths": truth, "filenames": filenames,
        "elapsed_seconds": time.perf_counter() - started,
        "peak_allocated_bytes": torch.cuda.max_memory_allocated(device) if device.type == "cuda" else None,
        "peak_reserved_bytes": torch.cuda.max_memory_reserved(device) if device.type == "cuda" else None}


def selection_record(epoch, seen, unseen):
    if type(epoch) is not int or epoch < 0:
        raise ValueError("Selection requires a nonnegative epoch.")
    for result, expected in ((seen, 1238), (unseen, 1894)):
        count = result["sample_count"]
        if isinstance(count, bool) or not isinstance(count, Integral) or count != expected:
            raise ValueError("Selection requires the complete approved 1238 seen / 1894 unseen folds.")
    combined = (seen["sample_count"] * seen["loss"] + unseen["sample_count"] * unseen["loss"]) / (
        seen["sample_count"] + unseen["sample_count"])
    if not math.isfinite(combined):
        raise ValueError("Non-finite combined validation loss.")
    return {"epoch": epoch, "h1": harmonic_mean(seen["metrics"]["top1"], unseen["metrics"]["top1"]),
        "h5": harmonic_mean(seen["metrics"]["top5"], unseen["metrics"]["top5"]),
        "seen_top1": seen["metrics"]["top1"], "seen_top5": seen["metrics"]["top5"],
        "unseen_top1": unseen["metrics"]["top1"], "unseen_top5": unseen["metrics"]["top5"],
        "seen_count": seen["sample_count"], "unseen_count": unseen["sample_count"],
        "combined_validation_loss": combined, "selection_policy": dict(SELECTION_POLICY)}


def better_selection(candidate, best):
    def key(record):
        if (not math.isfinite(record["h1"]) or not 0 <= record["h1"] <= 1
                or not math.isfinite(record["combined_validation_loss"])
                or type(record["epoch"]) is not int or record["epoch"] < 0):
            raise ValueError("Invalid checkpoint-selection record.")
        return (-record["h1"], record["combined_validation_loss"], record["epoch"])
    candidate_key = key(candidate)
    return best is None or candidate_key < key(best)


## 7. Safe checkpoints, declared provenance and exact selection
Checkpoints store only the small adapter state, original direct text vectors, fixed prototypes, mappings, model/artifact/preprocessing/source/split/code identities and selection record. No CLIP backbone weights or optimizer-resume state is saved. Reconstruction uses weights_only=True and never loads/downloads CLIP. An explicit expected contract is required, preventing silent class/candidate reorder acceptance.


In [ ]:
def require_external_path(path):
    """Compatibility name: outputs now belong to the ignored canonical workspace."""
    path = Path(path)
    if not path.is_absolute():
        raise ValueError("Use an absolute canonical workspace path.")
    project = Path(CANONICAL_PROJECT_ROOT).resolve()
    workspace = Path(WORKSPACE_DIRECTORY)
    if workspace != project / "workspace" or workspace.resolve() != workspace:
        raise ValueError("The canonical workspace must not redirect outside the project.")
    allowed = workspace / "experiments"
    if allowed.resolve() != allowed:
        raise ValueError("The experiments directory must not redirect outside the workspace.")
    path = path.resolve()
    if path != allowed and allowed not in path.parents:
        raise ValueError("M2 outputs must stay inside canonical workspace/experiments/.")
    for directory in (path, *path.parents):
        if (directory / ".git").exists():
            raise ValueError("Experiment outputs must not enter a nested Git checkout.")
        if directory == allowed:
            break  # The enclosing canonical repository is deliberately permitted.
    return path

def save_adapter_checkpoint(path, adapter, text_state, contract, record):
    validate_text_state(text_state, contract)
    if type(record.get("epoch")) is not int or record["epoch"] < 0:
        raise ValueError("Checkpoint metadata must contain a nonnegative epoch.")
    if not isinstance(adapter, ResidualVisualAdapter):
        raise ValueError("Only the declared residual adapter can be checkpointed.")
    path = require_external_path(path)
    json.dumps(contract, allow_nan=False)
    json.dumps(record, allow_nan=False)
    if any(not torch.isfinite(t).all() for t in adapter.state_dict().values()):
        raise ValueError("Non-finite adapter parameters cannot be saved.")
    if adapter.residual_cap != contract["hyperparameters"]["residual_cap"]:
        raise ValueError("Adapter cap disagrees with the declared experiment.")
    payload = {"schema": "wp3b-visual-adapter-m4-v1", "adapter_spec": dict(contract["adapter"]),
        "adapter_state": {k: v.detach().cpu().clone() for k, v in adapter.state_dict().items()},
        "contract": contract, "contract_sha256": json_hash(contract),
        "text_state": {k: (v.detach().cpu().clone() if isinstance(v, torch.Tensor) else v)
                       for k, v in text_state.items()},
        "selection": record}
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("xb") as handle:  # Never overwrite an existing checkpoint.
        torch.save(payload, handle)
    return str(path)


def load_adapter_checkpoint(path, expected_contract):
    payload = torch.load(Path(path), map_location="cpu", weights_only=True)
    if payload.get("schema") != "wp3b-visual-adapter-m4-v1" or payload.get("adapter_spec") != expected_contract["adapter"]:
        raise ValueError("Unsupported adapter schema or architecture.")
    if (payload["contract_sha256"] != json_hash(payload["contract"])
            or payload["contract_sha256"] != json_hash(expected_contract)
            or payload["contract"] != expected_contract):
        raise ValueError("Model/source/split/candidate/component/preprocessing contract mismatch.")
    validate_text_state(payload["text_state"], expected_contract)
    adapter = ResidualVisualAdapter(expected_contract["hyperparameters"]["residual_cap"])
    adapter.load_state_dict(payload["adapter_state"], strict=True)
    if any(not torch.isfinite(t).all() for t in adapter.state_dict().values()):
        raise ValueError("Non-finite adapter checkpoint.")
    adapter.eval()
    # This reconstructs the small adapter only; it never loads/downloads CLIP.
    return adapter, payload["text_state"], payload["selection"]


def verify_split_identity(split):
    identity = split_identity(split)
    if sha256_file(split.source_csv) != identity["source_sha256"]:
        raise ValueError("Training-source bytes changed.")
    if any(identity[k] != v for k, v in EXPECTED_IDENTITIES.items()):
        raise ValueError("Historical source/split/candidate identity changed.")
    if [len(split.fit), len(split.val_seen), len(split.val_unseen)] != [4868, 1238, 1894]:
        raise ValueError("Unexpected development partition sizes.")
    return identity


def notebook_algorithm_hash():
    book = json.loads((ROOT / "notebooks/wp3b_bounded_cached.ipynb").read_text())
    nodes = [ast.dump(n, include_attributes=False) for c in book["cells"] if c["cell_type"] == "code"
             for n in ast.parse("".join(c["source"])).body if isinstance(n, (ast.FunctionDef, ast.ClassDef))]
    return json_hash(nodes)


def make_m4_contract(split, table, config):
    if config["model"] != "ViT-L/14" or config["seed"] != 42:
        raise ValueError("M2 fixes ViT-L/14 and seed 42.")
    for name in ("batch_size", "text_batch_size", "epochs"):
        if type(config[name]) is not int or config[name] < 1:
            raise ValueError("Positive integer budgets required.")
    if type(config["num_workers"]) is not int or config["num_workers"] < 0:
        raise ValueError("num_workers must be nonnegative.")
    for name in ("learning_rate", "weight_decay"):
        value = config[name]
        if isinstance(value, bool) or not isinstance(value, Real) or not math.isfinite(value):
            raise ValueError("Finite optimizer settings required.")
        if value < 0 or (name == "learning_rate" and value == 0):
            raise ValueError("Invalid optimizer setting.")
    if torch.device(config["device"]).type not in ("cpu", "cuda"):
        raise ValueError("Only CPU/CUDA devices are supported.")
    for key in ("image_directory", "weight_directory", "output_directory"):
        if not Path(config[key]).is_absolute():
            raise ValueError("Explicit absolute WSL paths required.")
    if config["epochs"] > 5:
        raise ValueError("The initial pilot is capped at five epochs.")
    fixed = {"batch_size": 8, "text_batch_size": 32, "num_workers": 0, "epochs": 5,
             "learning_rate": 1e-4, "weight_decay": 1e-4}
    if any(config.get(k) != v for k, v in fixed.items()):
        raise ValueError("The paired pilot fixes batch, optimizer and five-epoch budget.")
    if type(config.get("conditional_weight")) not in (int, float) or config["conditional_weight"] not in (0., 1.):
        raise ValueError("Select only the declared control (0) or intervention (1).")
    require_external_path(config["output_directory"])
    if config["conditional_weight"] != 1.0 or config.get("residual_cap") not in (None, .2):
        raise ValueError("M4 keeps M3 loss fixed and compares only unbounded versus rho=.20.")
    if isinstance(config.get("residual_cap"), bool):
        raise ValueError("Residual cap cannot be Boolean.")
    if not config.get("feature_cache_path") or not Path(config["feature_cache_path"]).is_absolute():
        raise ValueError("Declare an explicit verified shared-cache path.")
    require_external_path(config["feature_cache_path"])
    fingerprint = config.get("feature_cache_sha256")
    if not isinstance(fingerprint, str) or len(fingerprint) != 64 or any(x not in "0123456789abcdef" for x in fingerprint):
        raise ValueError("An explicit verified cache SHA-256 is required.")
    expected_table = build_prompt_table(split)
    if not table.equals(expected_table):
        raise ValueError("Candidate/prompt ordering changed.")
    prompt_hash = json_hash({"policy": PROMPT_POLICY, "rows": table.to_dict("records")})
    if prompt_hash != EXPECTED_DIRECT_PROMPT_SHA256:
        raise ValueError("The accepted direct prompt policy changed.")
    if LOSS_SPEC != {"crop_weight": .5, "disease_weight": .5, "identity_weight": 1.,
                     "temperature": .07, "identity": "mean sample squared L2 between unit adapted and unit frozen features"}:
        raise ValueError("M2 has one fixed predeclared objective.")
    return {"schema": "wp3b-m4-contract-v1", "split": verify_split_identity(split),
        "fitting_composition_ids": sorted(int(x) for x in split.fit.composition_id.unique()),
        "candidate_ids": list(split.candidate_ids), "candidate_pairs": [list(x) for x in split.candidate_pairs],
        "component_ids": {k: [int(x) for x in sorted(table[k + "_id"].unique())] for k in ("crop", "disease")},
        "component_names": {k: [[int(i), str(n)] for i, n in sorted(split.mappings[k + "_id_to_name"].items())]
                            for k in ("crop", "disease")},
        "direct_prompt_policy": dict(PROMPT_POLICY), "direct_prompt_sha256": prompt_hash,
        "direct_prompt_records": table.to_dict("records"), "model": {"name": "ViT-L/14",
            "revision": CLIP_REVISION, **weight_spec("ViT-L/14")},
        "adapter": {**ADAPTER_SPEC, "residual_cap": config["residual_cap"]}, "residual_bound": dict(RESIDUAL_CAP_SPEC), "component_prototypes": dict(PROTOTYPE_SPEC),
        "conditional_objective": dict(M3_SPEC), "loss": dict(LOSS_SPEC), "preprocessing": dict(NATIVE_PREPROCESSING),
        "hyperparameters": dict(config), "optimizer": "AdamW; adapter only; fixed LR; no AMP/scheduler",
        "gradient_norm_clip": 1., "selection_policy": dict(SELECTION_POLICY),
        "code": {"WP3A_parent": WP3A_PARENT, "algorithm_sha256": notebook_algorithm_hash()},
        "environment": {"python": platform.python_version(), "torch": str(torch.__version__),
                        "torchvision": str(torchvision.__version__)}}


def declare_m4_run(contract, output_directory):
    root = require_external_path(output_directory)
    root.mkdir(parents=True, exist_ok=True)
    directory = root / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:12])
    directory.mkdir(exist_ok=False)
    with (directory / "declaration.json").open("x", encoding="utf-8") as handle:
        json.dump({"contract": contract, "contract_sha256": json_hash(contract),
                   "predeclared_utc": datetime.now(timezone.utc).isoformat()}, handle, indent=2, allow_nan=False)
    pd.DataFrame(contract["direct_prompt_records"]).to_csv(directory / "direct_prompts.csv", index=False, mode="x")
    return directory


def verify_prepared_state(split, table, config, contract, run_directory, model, model_provenance, state):
    if make_m4_contract(split, table, config) != contract:
        raise ValueError("Configuration/source/code changed after declaration.")
    declaration = json.loads((Path(run_directory) / "declaration.json").read_text())
    if declaration["contract"] != contract or declaration["contract_sha256"] != json_hash(contract):
        raise ValueError("Missing or changed on-disk declaration.")
    validate_feature_cache(FEATURE_CACHE, split, table, config)
    verify_cache_image_files(FEATURE_CACHE, config)
    if sha256_file(require_external_path(config["feature_cache_path"])) != config["feature_cache_sha256"]:
        raise ValueError("Cache artifact changed after declaration.")
    require_frozen_clip(model)
    if (model_provenance["model"] != "ViT-L/14" or model_provenance["revision"] != CLIP_REVISION
            or model_provenance["artifact"]["sha256"] != contract["model"]["sha256"]):
        raise ValueError("CLIP provenance mismatch.")
    if canonical_native_preprocess_repr(model_provenance["native_preprocess"]) != native_preprocess_repr():
        raise ValueError("Expected native CLIP preprocessing.")
    validate_text_state(state, contract)
    return True


## 8. Owner-only pilot and result export
Epoch zero is a selectable identity control. Five fitting epochs follow only after explicit owner enablement; every epoch validates the same two folds. The exact selection order is H@1, combined sample-weighted validation objective, then earliest epoch. No tolerance or early-stopping rule is added. A later loss-only tie-break is not evidence of improved H@1.


In [ ]:
def evaluate_m4_folds(split, model, adapter, preprocess, state, config):
    results = {}
    for role in ("val_seen", "val_unseen"):
        loader = make_m4_loader(split, role, preprocess, config)
        results[role] = validate_adapter(model, adapter, loader, state, getattr(split, role),
                                         split.candidate_ids, config["device"])
    return results


def history_row(epoch, train, validation):
    seen, unseen = validation["val_seen"], validation["val_unseen"]
    row = selection_record(epoch, seen, unseen)
    row = {k: v for k, v in row.items() if k != "selection_policy"}
    for role, result in validation.items():
        for key in ("loss", "crop_loss", "disease_loss", "identity_loss", "elapsed_seconds",
                    "peak_allocated_bytes", "peak_reserved_bytes"):
            row[role + "_" + key] = result[key]
    for key in ("loss", "crop_loss", "disease_loss", "identity_loss", "elapsed_seconds",
                "peak_allocated_bytes", "peak_reserved_bytes"):
        row["train_" + key] = None if train is None else train[key]
    row["train_sample_count"] = None if train is None else train["sample_count"]
    row["train_conditional_loss"] = None if train is None else train["conditional_loss"]
    return row


def new_checkpoint_path(run_directory, epoch):
    return Path(run_directory) / ("adapter_epoch_" + str(epoch).zfill(3) + "_" + uuid.uuid4().hex[:8] + ".pt")


def run_adapter_pilot(split, model, adapter, preprocess, state, config, contract, run_directory,
                      model_provenance, baseline_record, baseline_validation, baseline_checkpoint):
    verify_prepared_state(split, build_prompt_table(split), config, contract, run_directory,
                          model, model_provenance, state)
    if baseline_record is None or baseline_record["epoch"] != 0 or not Path(baseline_checkpoint).is_file():
        raise ValueError("Record and save the identity epoch-zero control before training.")
    if config["epochs"] > 5:
        raise ValueError("This initial manual pilot is capped at five adapter epochs.")
    if torch.count_nonzero(adapter.up.weight) or torch.count_nonzero(adapter.up.bias):
        raise ValueError("Start this pilot from fresh identity initialization; resumption is unsupported.")
    optimizer = make_adapter_optimizer(adapter, config)
    loader = make_m4_loader(split, "fit", preprocess, config)
    best, best_validation, best_checkpoint = baseline_record, baseline_validation, baseline_checkpoint
    history = [history_row(0, None, baseline_validation)]
    epoch_predictions = prediction_rows(baseline_validation, 0)
    for epoch in range(1, config["epochs"] + 1):
        train = train_adapter_epoch(model, adapter, loader, optimizer, state, split.fit, config["device"])
        validation = evaluate_m4_folds(split, model, adapter, preprocess, state, config)
        record = selection_record(epoch, validation["val_seen"], validation["val_unseen"])
        history.append(history_row(epoch, train, validation))
        epoch_predictions.extend(prediction_rows(validation, epoch))
        if better_selection(record, best):
            best = record
            best_validation = validation
            best_checkpoint = save_adapter_checkpoint(new_checkpoint_path(run_directory, epoch),
                                                      adapter, state, contract, record)
        print({"epoch": epoch, "train_loss": train["loss"], "validation": record,
               "selected_epoch": best["epoch"], "selected_checkpoint": best_checkpoint})
    return {"history": history, "epoch_predictions": epoch_predictions, "best_record": best, "best_validation": best_validation,
            "best_checkpoint": best_checkpoint, "baseline_record": baseline_record,
            "strict_H1_gain_over_identity": best["h1"] > baseline_record["h1"]}


def export_m4_results(run_directory, contract, pilot, split):
    directory = require_external_path(run_directory)
    targets = [directory / name for name in ("adapter_history.csv", "adapter_selected_predictions.csv", "adapter_summary.json", "adapter_epoch_predictions.csv")]
    if any(p.exists() for p in targets):
        raise FileExistsError("Results already exported; preserve them and use a fresh run.")
    rows = []
    for role, result in pilot["best_validation"].items():
        for i, name in enumerate(result["filenames"]):
            row = {"fold": role, "image_name": name, "truth_global_id": int(result["truths"][i])}
            row.update({"rank_" + str(k + 1): int(v) for k, v in enumerate(result["ranking"][i, :5])})
            rows.append(row)
    summary = {"contract_sha256": json_hash(contract), "selected": pilot["best_record"],
        "baseline": pilot["baseline_record"], "checkpoint": pilot["best_checkpoint"],
        "strict_H1_gain_over_identity": pilot["strict_H1_gain_over_identity"],
        "history": pilot["history"], "diagnostics": diagnose_predictions(pd.DataFrame(rows), split),
        "config": contract["hyperparameters"], "protocol": {key: contract[key] for key in ("split", "candidate_ids", "candidate_pairs",
            "direct_prompt_sha256", "model", "preprocessing", "code", "conditional_objective")},
        "optimizer_resumption_supported": False,
        "interpretation": "A loss-only tie-break does not establish an H@1 improvement; no significance claim."}
    serialized = json.dumps(summary, indent=2, allow_nan=False)
    pd.DataFrame(pilot["history"]).to_csv(targets[0], index=False, mode="x")
    pd.DataFrame(rows).to_csv(targets[1], index=False, mode="x")
    with targets[2].open("x", encoding="utf-8") as handle:
        handle.write(serialized)
    pd.DataFrame(pilot["epoch_predictions"]).to_csv(targets[3], index=False, mode="x")
    return [str(p) for p in targets]


## 8b. Transparent diagnostics and paired success gate
Complete filename-aligned saved ranks produce crop/disease agreement, unseen pair/healthy/fitting-preference diagnostics. No missing row is dropped. All six epochs export only top-five IDs, retaining epoch 1 for later analysis. A success claim requires H@1 improvement and recovery of at least frozen diseased 79/1,496, with fewer false-healthy errors than paired and historical M2 controls. This gate does not change checkpoint selection.


In [ ]:
def evaluate_historical_m2_epoch1(checkpoint_path, split, model, preprocess, config, model_provenance):
    """Owner-only full development inference; never a training initializer."""
    information=inspect_saved_text_geometry(checkpoint_path,split)
    if information["selected_epoch"] != 1:
        raise ValueError("Supply the preserved historical epoch-1 checkpoint.")
    payload=torch.load(Path(checkpoint_path),map_location="cpu",weights_only=True)
    contract=payload["contract"]
    require_frozen_clip(model)
    if (model_provenance["model"] != "ViT-L/14" or model_provenance["revision"] != contract["model"]["revision"]
            or model_provenance["artifact"]["sha256"] != contract["model"]["sha256"]
            or contract["preprocessing"] != NATIVE_PREPROCESSING):
        raise ValueError("Historical checkpoint and frozen backbone/preprocessing differ.")
    adapter=ResidualVisualAdapter()
    adapter.load_state_dict(payload["adapter_state"],strict=True)
    if any(not torch.isfinite(v).all() for v in adapter.state_dict().values()):
        raise ValueError("Non-finite historical adapter weights.")
    adapter.to(config["device"]).eval()
    validation=evaluate_m4_folds(split,model,adapter,preprocess,payload["text_state"],config)
    return {"checkpoint_sha256":information["checkpoint_sha256"],"original_selection":payload["selection"],
            "validation":validation,"diagnostics":diagnose_predictions(pd.DataFrame(prediction_rows(validation,1)),split)}


def inspect_saved_text_geometry(checkpoint_path, split):
    """CPU analysis of saved text tensors/provenance, not new CLIP inference."""
    payload = torch.load(Path(checkpoint_path), map_location="cpu", weights_only=True)
    if payload.get("schema") != "wp3b-visual-adapter-v1":
        raise ValueError("Expected an immutable historical M2 adapter checkpoint.")
    contract=payload["contract"];state=payload["text_state"]
    if (payload["contract_sha256"] != json_hash(contract)
            or contract["split"] != verify_split_identity(split)
            or contract["candidate_ids"] != list(split.candidate_ids)
            or contract["direct_prompt_sha256"] != EXPECTED_DIRECT_PROMPT_SHA256
            or state["contract_sha256"] != payload["contract_sha256"]):
        raise ValueError("Historical checkpoint provenance mismatch.")
    for key in ("direct","crop","disease"):
        if tensor_sha256(state[key]) != state["hashes"][key]:
            raise ValueError("Historical saved text tensor hash mismatch.")
    pool=sorted(int(i) for i in split.fit.composition_id.unique())
    if contract["fitting_composition_ids"] != pool or state["prototype_pool_ids"] != pool:
        raise ValueError("Historical prototypes are not fitting-only.")
    positions=[contract["candidate_ids"].index(key) for key in pool]
    pairs=torch.tensor([contract["candidate_pairs"][i] for i in positions])
    vectors=unit_features(state["direct"][positions]);similarity=vectors@vectors.T
    identity=torch.eye(len(pool),dtype=torch.bool)
    same_crop=pairs[:,0,None]==pairs[None,:,0]
    same_disease=pairs[:,1,None]==pairs[None,:,1]
    statistics={}
    for name,mask in (("same_crop_different_disease",same_crop&~same_disease&~identity),
                      ("same_disease_different_crop",same_disease&~same_crop&~identity)):
        values=similarity[mask]
        statistics[name]={"ordered_pairs":len(values),"mean_cosine":float(values.mean()) if len(values) else None}
    healthy={int(row.disease_id) for row in split.fit.itertuples() if normalize_label(row.disease_name)=="healthy"}
    table=split.fit.drop_duplicates("composition_id")
    counts=split.fit.disease_id.value_counts()
    by_crop=table.groupby("crop_id").size()
    healthy_crops=set(table.loc[table.disease_id.isin(healthy),"crop_id"])
    return {"checkpoint_sha256":sha256_file(checkpoint_path),"contract_sha256":payload["contract_sha256"],
        "selected_epoch":payload["selection"]["epoch"],"fitting_images":len(split.fit),
        "healthy_images":int(split.fit.disease_id.isin(healthy).sum()),
        "disease_min_median_max":[int(counts.min()),float(counts.median()),int(counts.max())],
        "diseases_in_one_crop":int((table.groupby("disease_id").crop_id.nunique()==1).sum()),
        "conditional_eligible_images":int(split.fit.crop_id.isin(by_crop[by_crop>1].index).sum()),
        "healthy_negative_eligible_diseased_images":int((~split.fit.disease_id.isin(healthy)&split.fit.crop_id.isin(healthy_crops)).sum()),
        "text_geometry":statistics,
        "interpretation":"Descriptive saved-text/fitting evidence; crop confounding is a hypothesis, not a proven cause."}


def prediction_rows(validation, epoch):
    rows = []
    for role, result in validation.items():
        for i, name in enumerate(result["filenames"]):
            row = {"epoch": int(epoch), "fold": role, "image_name": name,
                   "truth_global_id": int(result["truths"][i])}
            row.update({"rank_" + str(k+1): int(v) for k, v in enumerate(result["ranking"][i, :5])})
            row.update({"control_rank_" + str(k+1): int(v) for k, v in enumerate(result["control_ranking"][i, :5])})
            rows.append(row)
    return rows


def diagnose_predictions(predictions, split):
    """Counts from complete saved development ranks only; no image/model access."""
    expected = pd.concat([split.val_seen.assign(fold="val_seen"),
                          split.val_unseen.assign(fold="val_unseen")], ignore_index=True)
    if (predictions.duplicated(["fold", "image_name"]).any() or len(predictions) != len(expected)
            or set(zip(predictions.fold, predictions.image_name)) != set(zip(expected.fold, expected.image_name))):
        raise ValueError("Require each development filename exactly once; missing rows are not dropped.")
    frame = expected.merge(predictions, on=["fold", "image_name"], validate="one_to_one")
    if not frame.composition_id.eq(frame.truth_global_id).all():
        raise ValueError("Saved truths differ from the approved training-derived manifest.")
    pair = dict(zip(split.candidate_ids, split.candidate_pairs))
    for column in ["truth_global_id"] + ["rank_"+str(k) for k in range(1,6)]:
        values = frame[column]
        if values.dtype.kind not in "iu" or not set(values) <= set(pair):
            raise ValueError("Prediction values must be original integer global IDs.")
    if any(len(set(row)) != 5 for row in frame[["rank_"+str(k) for k in range(1,6)]].itertuples(index=False,name=None)):
        raise ValueError("Duplicate ranked candidate IDs.")
    healthy = {int(row.disease_id) for row in split.fit.itertuples()
               if normalize_label(row.disease_name) == "healthy"}
    predicted = frame.rank_1.map(pair)
    frame["pred_crop"] = predicted.map(lambda value:value[0])
    frame["pred_disease"] = predicted.map(lambda value:value[1])
    frame["correct1"] = frame.rank_1.eq(frame.composition_id)
    frame["correct5"] = frame[["rank_"+str(k) for k in range(1,6)]].eq(frame.composition_id,axis=0).any(axis=1)
    frame["true_healthy"] = frame.disease_id.isin(healthy)
    frame["pred_healthy"] = frame.pred_disease.isin(healthy)
    def counts(part):
        return {"count":len(part),"correct1":int(part.correct1.sum()),"correct5":int(part.correct5.sum()),
            "top1":float(part.correct1.mean()) if len(part) else None,
            "top5":float(part.correct5.mean()) if len(part) else None,
            "crop_correct":int(part.pred_crop.eq(part.crop_id).sum()),
            "disease_correct":int(part.pred_disease.eq(part.disease_id).sum()),
            "predicted_healthy":int(part.pred_healthy.sum()),
            "healthy_count":int(part.true_healthy.sum()),
            "healthy_correct":int((part.correct1 & part.true_healthy).sum()),
            "diseased_count":int((~part.true_healthy).sum()),
            "diseased_correct":int((part.correct1 & ~part.true_healthy).sum()),
            "false_healthy":int((part.pred_healthy & ~part.true_healthy).sum()),
            "predicted_fitting_composition":int(part.rank_1.isin(split.fit.composition_id.unique()).sum())}
    result = {role:counts(part) for role,part in frame.groupby("fold")}
    result["per_pair"] = [{"fold":role,"global_composition_id":int(key),**counts(part)}
                          for (role,key),part in frame.groupby(["fold","composition_id"])]
    result["per_crop"] = [{"fold":role,"crop":crop,**counts(part)}
                          for (role,crop),part in frame.groupby(["fold","crop_name"])]
    result["per_disease"] = [{"fold":role,"disease_id":int(disease),**counts(part)}
                             for (role,disease),part in frame.groupby(["fold","disease_id"])]
    result["focus_crops"] = [{"fold":role,"crop":crop,**counts(part)}
                             for (role,crop),part in frame[frame.crop_name.isin(["tomato","cassava"])].groupby(["fold","crop_name"])]
    return result


def paired_correctness(first, second):
    keys = ["fold", "image_name", "truth_global_id"]
    for table in (first, second):
        if table.duplicated(["fold", "image_name"]).any():
            raise ValueError("Duplicate prediction identities.")
    if set(map(tuple,first[keys].to_numpy())) != set(map(tuple,second[keys].to_numpy())):
        raise ValueError("Paired records require identical filenames/folds/truths.")
    joined = first[keys+["rank_1"]].merge(second[keys+["rank_1"]],on=keys,validate="one_to_one")
    result = {}
    for role,part in joined.groupby("fold"):
        a=part.rank_1_x.eq(part.truth_global_id);b=part.rank_1_y.eq(part.truth_global_id)
        result[role]={"count":len(part),"both_correct":int((a&b).sum()),"first_only":int((a&~b).sum()),
                      "second_only":int((~a&b).sum()),"both_wrong":int((~a&~b).sum())}
    return result


def compare_pilot_summaries(control, intervention):
    if control["protocol"] != intervention["protocol"]:
        raise ValueError("Paired candidate/source/split/model/code protocols differ.")
    if control["config"]["residual_cap"] is not None or intervention["config"]["residual_cap"] != .2:
        raise ValueError("Compare only the unbounded control and rho=.20 intervention.")
    stripped=lambda cfg:{k:v for k,v in cfg.items() if k not in ("residual_cap","output_directory")}
    if stripped(control["config"])!=stripped(intervention["config"]):
        raise ValueError("Both arms require identical loss, cache, optimizer and budgets.")
    for summary in (control,intervention):
        if [row["epoch"] for row in summary["history"]]!=list(range(6)):
            raise ValueError("Both five-epoch budgets plus epoch zero must be complete.")
        if any(row.get("train_sample_count")!=4868 for row in summary["history"][1:]):
            raise ValueError("Each fitting epoch must process all 4868 samples.")
        for role,expected in (("val_seen",1238),("val_unseen",1894)):
            d=summary["diagnostics"][role];key="seen_top1" if role=="val_seen" else "unseen_top1"
            if d["count"]!=expected or summary["selected"][key]!=d["correct1"]/expected:
                raise ValueError("Metric and full processed counts disagree.")
        if summary["selected"]["h1"]!=harmonic_mean(summary["selected"]["seen_top1"],summary["selected"]["unseen_top1"]):
            raise ValueError("Stored H@1 disagrees with fold accuracies.")
    a=control["diagnostics"]["val_unseen"];b=intervention["diagnostics"]["val_unseen"]
    gain=intervention["selected"]["h1"]>max(control["selected"]["h1"],.1464606922871855)
    disease=b["diseased_correct"]>max(a["diseased_correct"],71)
    false_healthy=b["false_healthy"]<min(a["false_healthy"],490)
    return {"strict_H1_gain":gain,"improved_unseen_diseased_hits":disease,
        "reduced_false_healthy":false_healthy,"frozen_diseased_79_recovered":b["diseased_correct"]>=79,
        "supports_exploratory_success":gain and disease and false_healthy and b["diseased_correct"]>=79,
        "interpretation":"One fixed development pilot; no significance or final-test claim."}


def inspect_historical_records(workspace, split):
    """Read immutable saved predictions/history; no image/model execution."""
    workspace=Path(workspace)
    m2=workspace/"experiments/wp3b-m2/20261104T042104Z_a4708eb25dca"
    m1=workspace/"experiments/wp3b-m1/20261102T075353Z_b017e93531c2"
    old=pd.read_csv(m1/"paired_predictions.csv")
    selected=pd.read_csv(m2/"adapter_selected_predictions.csv")
    frozen=old[["fold","image_name","truth_global_id"]+["direct_rank_"+str(k) for k in range(1,6)]].rename(
        columns={"direct_rank_"+str(k):"rank_"+str(k) for k in range(1,6)})
    factorized=old[["fold","image_name","truth_global_id"]+["factorized_rank_"+str(k) for k in range(1,6)]].rename(
        columns={"factorized_rank_"+str(k):"rank_"+str(k) for k in range(1,6)})
    history=pd.read_csv(m2/"adapter_history.csv")
    return {"frozen":diagnose_predictions(frozen,split),"M1":diagnose_predictions(factorized,split),
            "M2_selected":diagnose_predictions(selected,split),"paired_frozen_M2":paired_correctness(frozen,selected),
            "M2_epoch_history":history.astype(object).where(pd.notna(history), None).to_dict("records"),
            "epoch1_limitation":"Aggregate history/checkpoint exists; per-image records were not saved. Owner-only inference needed later."}


## Shared frozen-feature cache
Both arms use the same verified raw features. Encoding happens once, explicitly, over 8,000 training-source images only. No augmentation is cached; the native transform is deterministic. Full hashes bind source/split/candidates/prompts, model, native preprocessing, labels, filenames, raw tensors and image bytes. A cached identity model enables the unchanged explicit epoch loops to train only the adapter. Cache loading never loads CLIP or opens image files. Historical Top-5 parity is checked before declaring a run; discrepancies stop execution.


In [ ]:
def inspect_completed_m3(split):
    """Independently re-count saved M3 predictions/checkpoints without image inference."""
    root=Path(WORKSPACE_DIRECTORY)/"experiments/wp3b-m3"
    output={}
    for arm,name in (("control","20261104T094921Z_365ee1b25a43"),
                     ("conditional","20261104T123450Z_9cdf4e1ddf6b")):
        folder=root/name
        summary=json.loads((folder/"adapter_summary.json").read_text())
        declaration=json.loads((folder/"declaration.json").read_text())
        contract=declaration["contract"]
        if declaration["contract_sha256"]!=json_hash(contract) or summary["contract_sha256"]!=json_hash(contract):
            raise ValueError("Historical M3 declaration/summary mismatch.")
        if contract["split"]!=verify_split_identity(split) or contract["candidate_ids"]!=list(split.candidate_ids):
            raise ValueError("Historical M3 source/candidate protocol changed.")
        all_rows=pd.read_csv(folder/"adapter_epoch_predictions.csv")
        if set(all_rows.epoch)!=set(range(6)) or len(all_rows)!=6*3132:
            raise ValueError("M3 requires all six complete epoch prediction records.")
        epochs=[]
        for epoch,rows in all_rows.groupby("epoch",sort=True):
            diagnostics=diagnose_predictions(rows,split)
            seen=diagnostics["val_seen"];unseen=diagnostics["val_unseen"]
            h1=harmonic_mean(seen["top1"],unseen["top1"])
            h5=harmonic_mean(seen["top5"],unseen["top5"])
            record=next(r for r in summary["history"] if r["epoch"]==int(epoch))
            if record["h1"]!=h1 or record["h5"]!=h5:
                raise ValueError("Recorded M3 history disagrees with per-image hit counts.")
            if epoch and record.get("train_sample_count")!=4868:
                raise ValueError("Incomplete fitting budget.")
            epochs.append({"epoch":int(epoch),"h1":h1,"h5":h5,"unseen":unseen})
        best=None
        for row in summary["history"]:
            if better_selection(row,best):best=row
        if best["epoch"]!=summary["selected"]["epoch"]:
            raise ValueError("Historical M3 selection is inconsistent.")
        selected=diagnose_predictions(pd.read_csv(folder/"adapter_selected_predictions.csv"),split)
        checkpoint=Path(summary["checkpoint"])
        payload=torch.load(checkpoint,map_location="cpu",weights_only=True)
        if payload["contract"]!=contract or payload["selection"]!=summary["selected"]:
            raise ValueError("Historical M3 selected checkpoint provenance mismatch.")
        output[arm]={"run_id":name,"selected":summary["selected"],"diagnostics":selected,"epochs":epochs,
                     "checkpoint_sha256":sha256_file(checkpoint),"contract_sha256":json_hash(contract)}
    return output


def canonical_native_preprocess_repr(value):
    """Remove only CLIP's process-local RGB-converter address; retain all transform settings."""
    import re
    if not isinstance(value, str):
        raise ValueError("Native preprocessing provenance must be a string.")
    return re.sub(r"<function _convert_image_to_rgb at 0x[0-9a-fA-F]+>",
                  "<function _convert_image_to_rgb>", value)


def native_preprocess_repr():
    from clip.clip import _transform
    return canonical_native_preprocess_repr(repr(_transform(224)))


def expected_cache_spec(split, table, config):
    if config["model"] != "ViT-L/14" or not table.equals(build_prompt_table(split)):
        raise ValueError("Only the fixed native ViT-L/14 development cache is supported.")
    prompt_hash = json_hash({"policy":PROMPT_POLICY,"rows":table.to_dict("records")})
    if prompt_hash != EXPECTED_DIRECT_PROMPT_SHA256:
        raise ValueError("Prompt policy/order changed.")
    return {"version":"native-frozen-clip-raw-f32-v1", "split":verify_split_identity(split),
        "candidate_ids":list(split.candidate_ids),"candidate_pairs":[list(v) for v in split.candidate_pairs],
        "model":"ViT-L/14","model_sha256":MODEL_REGISTRY["ViT-L/14"]["sha256"],
        "CLIP_revision":CLIP_REVISION,"prompt_sha256":prompt_hash,
        "preprocessing":dict(NATIVE_PREPROCESSING),"native_transform":native_preprocess_repr(),
        "image_embedding_dimension":768,"stored_dtype":"torch.float32; raw, not unit-normalized",
        "encoding_batch_size":8,"augmentation":None}


@torch.no_grad()
def build_feature_cache(model, preprocess, provenance, split, table, config, direct, progress=True):
    """Owner-only one-time image encoding. Each fold is kept separate and ordered."""
    from tqdm.auto import tqdm
    require_frozen_clip(model)
    spec=expected_cache_spec(split,table,config)
    if (provenance["model"] != spec["model"] or provenance["revision"] != CLIP_REVISION
            or provenance["artifact"]["sha256"] != spec["model_sha256"]
            or canonical_native_preprocess_repr(provenance["native_preprocess"]) != spec["native_transform"]
            or provenance["trainable_parameters"] != 0):
        raise ValueError("The native frozen model/preprocessing provenance differs.")
    folds={};image_hashes={};started=time.perf_counter()
    for role in ("fit","val_seen","val_unseen"):
        frame=getattr(split,role)
        dataset=CLIPDevelopmentDataset(frame,config["image_directory"],preprocess,set(frame.image_name))
        loader=DataLoader(dataset,batch_size=8,shuffle=False,num_workers=0,drop_last=False)
        embeddings=[];filenames=[]
        for images,targets in tqdm(loader,desc="Cache "+role,disable=not progress):
            vectors=model.encode_image(images.to(config["device"])).detach().float().cpu().clone()
            if vectors.shape != (len(images),768) or not torch.isfinite(vectors).all():
                raise ValueError("Unexpected native frozen image features.")
            embeddings.append(vectors);filenames.extend(targets["image_name"])
        if filenames != frame.image_name.tolist():
            raise ValueError("Cache extraction changed ordered identities.")
        for name in filenames:
            path=(Path(config["image_directory"])/name).resolve()
            if path.parent != Path(config["image_directory"]).resolve():
                raise ValueError("Cache image escapes the allowed directory.")
            image_hashes[name]=sha256_file(path)
        features=torch.cat(embeddings)
        labels=torch.tensor(frame[["crop_id","disease_id","composition_id"]].to_numpy(),dtype=torch.int64)
        folds[role]={"features":features,"features_sha256":tensor_sha256(features),
                     "filenames":filenames,"labels":labels,"labels_sha256":tensor_sha256(labels)}
    direct=direct.detach().cpu().clone()
    cache={"schema":"wp3b-native-clip-cache-v1","spec":spec,"spec_sha256":json_hash(spec),
        "folds":folds,"direct":direct,"direct_sha256":tensor_sha256(direct),
        "image_hashes":image_hashes,"image_hashes_sha256":json_hash(image_hashes),
        "model_provenance":provenance,"extraction_seconds":time.perf_counter()-started,
        "environment":{"python":platform.python_version(),"torch":str(torch.__version__),
                       "torchvision":str(torchvision.__version__)}}
    validate_feature_cache(cache,split,table,config)
    return cache


def validate_feature_cache(cache, split, table, config):
    expected=expected_cache_spec(split,table,config)
    stored_spec = cache.get("spec", {})
    comparable_spec = dict(stored_spec)
    if "native_transform" in comparable_spec:
        comparable_spec["native_transform"] = canonical_native_preprocess_repr(comparable_spec["native_transform"])
    # Validate the original raw digest before normalizing the legacy in-process repr.
    if (cache.get("schema") != "wp3b-native-clip-cache-v1" or comparable_spec != expected
            or cache.get("spec_sha256") != json_hash(stored_spec)):
        raise ValueError("Cache source/split/model/prompt/preprocessing contract mismatch.")
    if set(cache.get("folds",{})) != {"fit","val_seen","val_unseen"}:
        raise ValueError("Only the approved three training-derived folds are allowed.")
    provenance=cache["model_provenance"]
    if (provenance["model"] != "ViT-L/14" or provenance["revision"] != CLIP_REVISION
            or provenance["artifact"]["sha256"] != expected["model_sha256"]
            or canonical_native_preprocess_repr(provenance["native_preprocess"]) != expected["native_transform"]
            or provenance["trainable_parameters"] != 0):
        raise ValueError("Frozen cache model provenance changed.")
    names=[]
    for role in ("fit","val_seen","val_unseen"):
        frame=getattr(split,role);part=cache["folds"][role]
        labels=torch.tensor(frame[["crop_id","disease_id","composition_id"]].to_numpy(),dtype=torch.int64)
        x=part["features"]
        if (part["filenames"] != frame.image_name.tolist() or part["labels"].dtype != torch.int64
                or not torch.equal(part["labels"].cpu(),labels)
                or part["labels_sha256"] != tensor_sha256(labels)
                or x.shape != (len(frame),768) or x.dtype != torch.float32 or x.requires_grad
                or not torch.isfinite(x).all() or (torch.linalg.vector_norm(x,dim=1)<=0).any()
                or part["features_sha256"] != tensor_sha256(x)):
            raise ValueError("Cache row/tensor/label fingerprint changed: "+role)
        names.extend(part["filenames"])
    hashes=cache["image_hashes"]
    if (len(names)!=8000 or len(set(names))!=8000 or set(hashes)!=set(names)
            or any(not isinstance(v,str) or len(v)!=64 or any(c not in "0123456789abcdef" for c in v) for v in hashes.values())
            or cache["image_hashes_sha256"] != json_hash(hashes)):
        raise ValueError("Cache image identities/content provenance changed.")
    direct=cache["direct"]
    if (direct.shape!=(323,768) or direct.dtype!=torch.float64 or direct.requires_grad
            or not torch.isfinite(direct).all() or (torch.linalg.vector_norm(direct,dim=1)<=0).any()
            or cache["direct_sha256"]!=tensor_sha256(direct)):
        raise ValueError("Cache direct-text bank changed.")
    return True


def save_feature_cache(path, cache):
    path=require_external_path(path)
    path.parent.mkdir(parents=True,exist_ok=True)
    with path.open("xb") as handle:
        torch.save(cache,handle)
    return str(path)


def verify_cache_image_files(cache, config):
    """Re-hash only validated development filenames; never scan an image directory."""
    directory = Path(config["image_directory"]).resolve()
    for role in ("fit", "val_seen", "val_unseen"):
        for name in cache["folds"][role]["filenames"]:
            if (not isinstance(name, str) or Path(name).name != name
                    or any(character in name for character in ("/", "\\", ":"))):
                raise ValueError("Unsafe cached development filename.")
            path = (directory / name).resolve()
            if path.parent != directory or not path.is_file():
                raise ValueError("Missing/redirected approved development image: " + name)
            if sha256_file(path) != cache["image_hashes"][name]:
                raise ValueError("Current development image bytes differ from the cache: " + name)
    return True


def load_feature_cache(path, expected_sha256, split, table, config):
    path=require_external_path(path)
    if not path.is_file() or sha256_file(path)!=expected_sha256:
        raise ValueError("Missing cache or full artifact SHA-256 mismatch.")
    cache=torch.load(path,map_location="cpu",weights_only=True)
    validate_feature_cache(cache,split,table,config)
    verify_cache_image_files(cache,config)
    return cache


class CachedFeatureBackbone(torch.nn.Module):
    """Frozen identity boundary over previously verified raw CLIP embeddings."""
    def encode_image(self, features):
        if (features.ndim!=2 or features.shape[1]!=768 or not len(features)
                or not torch.isfinite(features).all()
                or (torch.linalg.vector_norm(features.float(),dim=1)<=0).any()):
            raise ValueError("Expected nonzero finite raw CLIP features [B,768].")
        return features.detach().float()


class CachedFeatureDataset(Dataset):
    def __init__(self,split,role,cache):
        if role not in ("fit","val_seen","val_unseen"):
            raise ValueError("Only training-derived cache roles are supported.")
        frame=getattr(split,role)
        self.role=role;self.frame=frame.copy();self.part=cache["folds"][role]
        labels=torch.tensor(frame[["crop_id","disease_id","composition_id"]].to_numpy(),dtype=torch.int64)
        if self.part["filenames"]!=frame.image_name.tolist() or not torch.equal(self.part["labels"],labels):
            raise ValueError("Cache dataset does not match the exact approved role.")
    def __len__(self):
        return len(self.frame)
    def __getitem__(self,index):
        c,d,p=self.part["labels"][index].tolist()
        return self.part["features"][index],{"crop":c,"disease":d,"composition":p,
                                          "image_name":self.part["filenames"][index]}


def verify_cached_frozen_reference(cache, split):
    """Owner-only cached scoring; stops on any historical Top-5 rank discrepancy."""
    path=Path(WORKSPACE_DIRECTORY)/"experiments/wp3b-m3/20261104T094921Z_365ee1b25a43/adapter_epoch_predictions.csv"
    reference=pd.read_csv(path)
    reference=reference.loc[reference.epoch==0]
    for role in ("val_seen","val_unseen"):
        part=cache["folds"][role];expected=reference.loc[reference.fold==role]
        if expected.image_name.tolist()!=part["filenames"]:
            raise ValueError("Historical frozen reference order differs.")
        ranks=rank_global_ids(cosine_candidate_scores(part["features"],cache["direct"]),split.candidate_ids)[:,:5]
        expected=torch.tensor(expected[["control_rank_"+str(k) for k in range(1,6)]].to_numpy())
        if not torch.equal(ranks,expected):
            raise ValueError("Cached/native frozen ranks differ. Preserve cache and investigate; no tolerance is silently applied.")
    return True


In [ ]:
def run_paired_cached_pilot(split, table, config):
    """Owner-invoked, exactly two fresh five-epoch arms; never called by default."""
    if FEATURE_CACHE is None:
        raise ValueError("Load the verified common cache before starting the paired pilot.")
    validate_feature_cache(FEATURE_CACHE,split,table,config)
    if sha256_file(require_external_path(config["feature_cache_path"])) != config["feature_cache_sha256"]:
        raise ValueError("Common cache artifact changed.")
    outcomes={}
    for label,cap in (("control",None),("bounded",.2)):
        arm={**config,"residual_cap":cap}
        contract=make_m4_contract(split,table,arm)
        directory=declare_m4_run(contract,arm["output_directory"])
        model=CachedFeatureBackbone().eval().to(arm["device"])
        provenance=FEATURE_CACHE["model_provenance"]
        prototypes=component_prototypes(FEATURE_CACHE["direct"],table,contract["fitting_composition_ids"])
        state=seal_text_state(FEATURE_CACHE["direct"],prototypes,contract)
        torch.manual_seed(arm["seed"])
        adapter=ResidualVisualAdapter(cap).to(arm["device"])
        verify_prepared_state(split,table,arm,contract,directory,model,provenance,state)
        baseline=evaluate_m4_folds(split,model,adapter,None,state,arm)
        if any(not torch.equal(v["ranking"],v["control_ranking"]) for v in baseline.values()):
            raise ValueError("Paired identity baseline differs from the frozen reference.")
        record=selection_record(0,baseline["val_seen"],baseline["val_unseen"])
        checkpoint=save_adapter_checkpoint(new_checkpoint_path(directory,0),adapter,state,contract,record)
        pilot=run_adapter_pilot(split,model,adapter,None,state,arm,contract,directory,
                                provenance,record,baseline,checkpoint)
        outputs=export_m4_results(directory,contract,pilot,split)
        summary_path=next(Path(path) for path in outputs if Path(path).name=="adapter_summary.json")
        summary=json.loads(summary_path.read_text())
        outcomes[label]={"directory":str(directory),"summary_path":str(summary_path),"summary":summary}
        print({"completed_arm":label,"selected":summary["selected"],"summary":str(summary_path)})
    decision=compare_pilot_summaries(outcomes["control"]["summary"],outcomes["bounded"]["summary"])
    print({"paired_decision":decision})
    return {"arms":outcomes,"decision":decision}


## Safe setup and one-time owner cache build
All default flags are False. Metadata inspection is cheap; no image or model is loaded by default. The owner explicitly builds one cache, then loads it and declares each fresh arm. The cache is common to both arms and does not depend on the cap. A cache hash/parity discrepancy is a stop condition, not permission to alter old results.


In [ ]:
SPLIT = build_development_split(ROOT / "DPD_Small_train.csv")
verify_split_identity(SPLIT)
DIRECT_TABLE = build_prompt_table(SPLIT)
print({"split":split_identity(SPLIT),"configuration":CONFIG,"residual_choices":RESIDUAL_CAP_SPEC})
display(DIRECT_TABLE.head(8))


In [ ]:
if BUILD_FEATURE_CACHE:
    if FEATURE_CACHE is not None or RUN_DIRECTORY is not None:
        raise RuntimeError("Build/load once in a fresh kernel before declaring an arm.")
    encoder, native_transform, provenance = load_frozen_clip(CONFIG)
    direct, text_provenance = encode_candidate_texts(encoder, DIRECT_TABLE, CONFIG)
    cache = build_feature_cache(encoder, native_transform, provenance, SPLIT, DIRECT_TABLE, CONFIG, direct)
    path = Path(CONFIG["output_directory"]) / "feature_cache" / (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:12] + ".pt")
    CONFIG["feature_cache_path"] = save_feature_cache(path, cache)
    CONFIG["feature_cache_sha256"] = sha256_file(path)
    print({"feature_cache_path":CONFIG["feature_cache_path"],"feature_cache_sha256":CONFIG["feature_cache_sha256"],
           "cache_bytes":path.stat().st_size,"actual_extraction_seconds":cache["extraction_seconds"]})
    # Release the large encoder; the trainable adapter will use only the verified features.
    encoder = None
    direct = None
    cache = None
else:
    print("BUILD_FEATURE_CACHE=False: no image encoding or weight loading.")

if LOAD_FEATURE_CACHE:
    if FEATURE_CACHE is not None or RUN_DIRECTORY is not None:
        raise RuntimeError("Load once before declaring the arm.")
    if CONFIG["feature_cache_path"] is None or CONFIG["feature_cache_sha256"] is None:
        raise ValueError("Supply the path and recorded SHA-256 from the one-time cache build.")
    FEATURE_CACHE = load_feature_cache(CONFIG["feature_cache_path"],CONFIG["feature_cache_sha256"],
                                      SPLIT,DIRECT_TABLE,CONFIG)
    verify_cached_frozen_reference(FEATURE_CACHE,SPLIT)
    print({"cache_verified":True,"historical_frozen_Top5_parity":True,
           "cached_folds":{k:list(v["features"].shape) for k,v in FEATURE_CACHE["folds"].items()}})
else:
    print("LOAD_FEATURE_CACHE=False: no cache file is loaded.")


## Recommended owner execution: one explicit paired run
Fresh kernel: enable BUILD_FEATURE_CACHE, LOAD_FEATURE_CACHE and RUN_PAIRED_PILOT only; leave all individual-run flags False. Run All builds/verifies one cache and then executes exactly the unbounded and bounded five-epoch arms sequentially, saving both sets of records and the checked comparison. To reuse an existing cache, keep BUILD_FEATURE_CACHE=False and supply its path/SHA. An interrupted attempt is never resumed or overwritten; diagnose it before rerunning. Default Run All remains metadata-only because all flags are False.


In [ ]:
if RUN_PAIRED_PILOT:
    if PREDECLARE_RUN or RUN_TRAINING or RUN_DIRECTORY is not None or PAIRED_RESULTS is not None or PAIRED_STARTED:
        raise RuntimeError("Use paired mode alone, once in a fresh kernel; preserve interrupted attempts.")
    PAIRED_STARTED = True
    PAIRED_RESULTS = run_paired_cached_pilot(SPLIT,DIRECT_TABLE,CONFIG)
    display(pd.DataFrame([{"arm":label,**item["summary"]["selected"]}
                         for label,item in PAIRED_RESULTS["arms"].items()]))
else:
    print("RUN_PAIRED_PILOT=False: the paired experiment is disabled.")


## Owner-declared arm and epoch-zero control
Choose CONFIG.residual_cap=None for the unbounded control, then .20 in a fresh kernel for the intervention. Both use conditional_weight=1 and exactly the same feature-cache path/hash, seed, batch and five epochs. Declaring a run seals the code/model/cache/data/objective contract before validation or optimization.


In [ ]:
if PREDECLARE_RUN:
    if RUN_DIRECTORY is not None or FEATURE_CACHE is None:
        raise RuntimeError("Load and validate the shared cache, then declare once.")
    CONTRACT = make_m4_contract(SPLIT,DIRECT_TABLE,CONFIG)
    RUN_DIRECTORY = declare_m4_run(CONTRACT,CONFIG["output_directory"])
    FROZEN_CLIP = CachedFeatureBackbone().eval().to(CONFIG["device"])
    MODEL_PROVENANCE = FEATURE_CACHE["model_provenance"]
    PREPROCESS = None  # Images were already processed by the verified native transform.
    prototypes = component_prototypes(FEATURE_CACHE["direct"],DIRECT_TABLE,CONTRACT["fitting_composition_ids"])
    TEXT_STATE = seal_text_state(FEATURE_CACHE["direct"],prototypes,CONTRACT)
    torch.manual_seed(CONFIG["seed"])
    ADAPTER = ResidualVisualAdapter(CONFIG["residual_cap"]).to(CONFIG["device"])
    print({"directory":str(RUN_DIRECTORY),"contract_sha256":json_hash(CONTRACT),
           "residual_cap":ADAPTER.residual_cap,"adapter_parameters":sum(p.numel() for p in ADAPTER.parameters())})
else:
    print("PREDECLARE_RUN=False: no experiment run created.")


In [ ]:
if RUN_DEVELOPMENT_EVALUATION and not RUN_TRAINING:
    if BASELINE_RECORD is not None:
        print("Identity baseline already exists in this kernel.")
    else:
        if ADAPTER is None or TEXT_STATE is None or not SAVE_CHECKPOINTS:
            raise RuntimeError("Declare the cache-based arm and explicitly enable checkpoint saving first.")
        if torch.count_nonzero(ADAPTER.up.weight) or torch.count_nonzero(ADAPTER.up.bias):
            raise RuntimeError("Epoch zero must use exact identity initialization.")
        verify_prepared_state(SPLIT,DIRECT_TABLE,CONFIG,CONTRACT,RUN_DIRECTORY,FROZEN_CLIP,MODEL_PROVENANCE,TEXT_STATE)
        BASELINE_VALIDATION = evaluate_m4_folds(SPLIT,FROZEN_CLIP,ADAPTER,PREPROCESS,TEXT_STATE,CONFIG)
        if any(not torch.equal(v["ranking"],v["control_ranking"]) for v in BASELINE_VALIDATION.values()):
            raise RuntimeError("Identity adapter differs from the frozen control.")
        BASELINE_RECORD = selection_record(0,BASELINE_VALIDATION["val_seen"],BASELINE_VALIDATION["val_unseen"])
        BASELINE_CHECKPOINT = save_adapter_checkpoint(new_checkpoint_path(RUN_DIRECTORY,0),ADAPTER,TEXT_STATE,CONTRACT,BASELINE_RECORD)
        print(BASELINE_RECORD)
else:
    print("Epoch-zero evaluation is disabled or training is the explicitly selected stage.")

if RUN_TRAINING:
    if TRAINING_STARTED or not RUN_DEVELOPMENT_EVALUATION or not SAVE_CHECKPOINTS or BASELINE_RECORD is None:
        raise RuntimeError("Fresh identity control, checkpoint saving and evaluation are required; never restart an interrupted attempt.")
    TRAINING_STARTED = True
    PILOT_RESULTS = run_adapter_pilot(SPLIT,FROZEN_CLIP,ADAPTER,PREPROCESS,TEXT_STATE,CONFIG,CONTRACT,
        RUN_DIRECTORY,MODEL_PROVENANCE,BASELINE_RECORD,BASELINE_VALIDATION,BASELINE_CHECKPOINT)
    display(pd.DataFrame(PILOT_RESULTS["history"]))
else:
    print("RUN_TRAINING=False: no adapter optimization.")


In [ ]:
if PILOT_RESULTS is not None:
    history = pd.DataFrame(PILOT_RESULTS["history"]).set_index("epoch")
    history[["train_loss","val_seen_loss","val_unseen_loss"]].plot(title="Actual cached-pilot losses")
    history[["h1","h5"]].plot(title="Actual development harmonic means")
    history[["val_seen_identity_loss","val_unseen_identity_loss"]].plot(title="Actual unit-feature drift")
    plt.show()
    print({"selected":PILOT_RESULTS["best_record"],"residual_cap":CONFIG["residual_cap"]})
elif PAIRED_RESULTS is not None:
    for label,item in PAIRED_RESULTS["arms"].items():
        history=pd.DataFrame(item["summary"]["history"]).set_index("epoch")
        history[["h1","h5"]].plot(title=label+" actual paired metrics")
    plt.show()
    print({"saved_paired_runs":{k:v["summary_path"] for k,v in PAIRED_RESULTS["arms"].items()},
           "decision":PAIRED_RESULTS["decision"]})
else:
    print("No experiment results exist; no plots are fabricated.")

if VERIFY_CHECKPOINT:
    if PILOT_RESULTS is None:raise RuntimeError("No checkpoint exists to verify.")
    restored,_,selected = load_adapter_checkpoint(PILOT_RESULTS["best_checkpoint"],CONTRACT)
    print({"restored_epoch":selected["epoch"],"residual_cap":restored.residual_cap})
if EXPORT_RESULTS:
    if PILOT_RESULTS is None:raise RuntimeError("No pilot exists to export.")
    print(export_m4_results(RUN_DIRECTORY,CONTRACT,PILOT_RESULTS,SPLIT))

if RUN_RECORD_DIAGNOSTICS:
    display(inspect_completed_m3(SPLIT))
if RUN_COMPARISON_REPORT:
    if CONTROL_SUMMARY_PATH is None or INTERVENTION_SUMMARY_PATH is None:
        raise ValueError("Supply both completed cache-based arm summaries.")
    first=json.loads(require_external_path(CONTROL_SUMMARY_PATH).read_text())
    second=json.loads(require_external_path(INTERVENTION_SUMMARY_PATH).read_text())
    print(compare_pilot_summaries(first,second))


## Exact manual sequence and scientific limits

Recommended simpler path: use the one-shot paired section above. Enable only BUILD_FEATURE_CACHE=True, LOAD_FEATURE_CACHE=True and RUN_PAIRED_PILOT=True in a fresh kernel; individual PREDECLARE_RUN/RUN_TRAINING/RUN_DEVELOPMENT_EVALUATION/SAVE_CHECKPOINTS flags stay False. Run All now explicitly performs the one native cache pass and exactly two five-epoch adapter arms, exports results and checks the gate. The steps below remain available for inspecting each stage separately.
1. Fresh WSL as04_yolo_katana kernel. For the first arm set CONFIG.residual_cap=None. Enable BUILD_FEATURE_CACHE=True and LOAD_FEATURE_CACHE=True, PREDECLARE_RUN=True, SAVE_CHECKPOINTS=True and RUN_DEVELOPMENT_EVALUATION=True; leave RUN_TRAINING=False. Run through the explicit cache/declaration/epoch-zero cells. Record the cache path/SHA printed. Building the cache is the only full native-CLIP image pass (8,000 training-derived images), no optimization. Do not use Run All with training enabled.
2. Set RUN_TRAINING=True in a new cell and execute only the owner-epoch cell. It trains exactly five cache-based adapter epochs. Enable VERIFY_CHECKPOINT and EXPORT_RESULTS in the live kernel and execute the export cell. Preserve the selected checkpoint and every epoch's Top-5 ranks. Do not rerun configuration/reset or declare cells in an existing attempt.
3. Fresh kernel: set CONFIG.residual_cap=.20, CONFIG.feature_cache_path and CONFIG.feature_cache_sha256 to the identical cache. BUILD_FEATURE_CACHE=False, LOAD_FEATURE_CACHE=True, then the same declaration/epoch-zero/five-epoch sequence. This does not reload CLIP weights or decode images. Start both arms from fresh identity adapters, not old M3 recovery states.
4. Supply both completed summary paths and enable RUN_COMPARISON_REPORT. It checks common protocol/cache, actual processed budgets and the predeclared H@1/disease/false-healthy gates. No new result is claimed until these runs exist.

The .20 relative norm bound implies an angle of at most asin(.20)=11.54 degrees, and nonzero output features. It is not a guarantee about healthy predictions or H@1. The unbounded control uses exact M3 forward arithmetic; cache extraction batching can still introduce native floating-point differences, which is why both arms share one cache and historical frozen Top-5 parity must pass. Cache reuse is valid only with the same frozen encoder, native deterministic preprocessing, image content and split; no augmentation, trainable CLIP or altered image resolution is supported. A checksum proves bytes/provenance consistency, not independent proof that an arbitrary cache was produced honestly. A changed image or cache pipeline requires a new cache and declaration.

All new runs remain under workspace/experiments/wp3b-m4. No official-test image/label development, repeated opportunistic tuning or silent historical selection changes. Primary H@1 selection stays exact, followed by sample-weighted base validation loss and earliest epoch. A scientific success additionally requires diseased hits above the paired control and historical M3, recovery of frozen79, and false-healthy errors below the paired control and historical M2's490. One seed remains exploratory. External ChatGPT review is no longer a required gate; automated verification precedes these owner-only expensive cells.
